# 1편 micrograd: 숫자 상자로 신경망과 역전파를 직접 만들기 (정답용)

단계마다 정본 코드가 들어 있습니다. 위에서부터 차례로 실행하면 모든 채점 셀이 "✓ 맞음"이 됩니다. 앞 셀을 고치는 단계에는 고친 뒤의 셀 전체와 다시 실행할 셀을 그 단계 자리에 다시 두었습니다.

출처: 코드는 Andrej Karpathy의 karpathy/micrograd, karpathy/nn-zero-to-hero(둘 다 MIT 라이선스)를 따라 만든 정본(inputs/pedagogy/content/L01.json) 그대로입니다. https://github.com/karpathy/micrograd · https://github.com/karpathy/nn-zero-to-hero

- 같은 노트북을 Colab과 맥미니 Jupyter 어느 쪽에서든 씁니다. 맨 위 환경 확인 셀이 실행 장소, 기기(cuda, mps, cpu 순으로 자동 선택), 그림 프로그램 graphviz(dot) 유무를 알려 줍니다. 1편 계산은 모두 cpu로 합니다.
- 설치가 필요할 때(정본 힌트 L01-S14-H2 그대로): 그림 도구 graphviz는 파이썬 부품과 컴퓨터 프로그램(dot) 두 가지가 다 있어야 합니다. Colab만: ModuleNotFoundError가 나면 새 셀에 `!pip install graphviz`를 실행한 뒤 준비 코드 셀을 다시 실행합니다. 맥미니 Jupyter: 저장소 가상환경에 파이썬 부품이, 맥에 graphviz 프로그램이 깔려 있어야 합니다. 없으면 직접 설치하지 말고 담당자에게 알리세요(설치는 승인 뒤, 환경 문제라 HELP로 표시). 그동안은 draw_dot(L) 대신 draw_text(L)로 같은 내용을 글자로 볼 수 있습니다. 기초 F-P8(에러 읽기)을 보세요.
- 채점 셀은 바로 위 셀이 보인 것(출력)이나 위에서 만든 이름을 봅니다. "○ 아직"이면 위 셀을 먼저 실행하세요. 그림 단계는 셀이 오류 없이 돌았는지만 보고, 그림은 '스스로 확인할 것' 목록으로 직접 확인합니다.
- 앞 셀을 고치는 단계는 그 셀을 그 자리에서 고친 뒤 '다시 실행할 셀'을 차례로 실행합니다. 그래서 다 끝낸 노트북을 처음부터 모두 실행하면, 고치기 전 모습을 보는 단계의 채점은 다르게 나올 수 있습니다. 결과코드는 단계를 해 나가며 받아 두세요.
- 맨 끝 "재구성" 구역은 본문을 마친 뒤(다른 날도 됩니다) 앞 단계를 보지 않고 다시 만들어 보는 연습입니다.

## 준비 1: 환경 확인

맨 처음 실행하세요(L01-S02 준비 코드). 우리가 새로 쓴 환경 확인 셀(설계서 3-0: 편 맨 앞에 기기 고르기 셀). 1편은 cpu로 충분해서 기기를 고르기만 하고 쓰지는 않습니다.

**보여야 하는 것**: 준비 코드 셀은 세 줄을 찍습니다: 실행 장소(Colab 또는 내 컴퓨터), 기기(cpu, mps, cuda 중 하나), 그림 프로그램 graphviz(dot) 있음 또는 없음. 그림 프로그램이 '없음'이면 S14의 힌트 H2를 미리 읽어 두세요. 5줄짜리 셀은 아무것도 출력되지 않으면 정상입니다. 셀 왼쪽에 실행 번호(예: [2])가 생기면 끝난 것입니다.

In [ ]:
import sys, shutil
import torch
in_colab = 'google.colab' in sys.modules
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('실행 장소:', 'Colab' if in_colab else '내 컴퓨터(맥미니 Jupyter 등)')
print('기기:', device, '(1편은 cpu로 충분합니다)')
print('그림 프로그램 graphviz(dot):', '있음' if shutil.which('dot') else '없음')

## 준비 2: 채점 도구

환경 확인 셀 다음에 한 번 실행하세요. 채점 도구(z2h_check)를 셀 하나에 담은 것이라 길고, 내용은 고치지 않습니다. 실행하면 `z2h_check 준비 완료` 한 줄이 보입니다. Colab에서 런타임을 다시 시작했다면 맨 위부터 다시 실행합니다.

In [ ]:
# z2h_check 묶음 0.2.0: Z2H 노트북 채점 도구를 셀 하나에 담은 것입니다.
# 이 셀을 그대로 실행하면 check(...)와 finish("L01")를 쓸 수 있습니다. 내용은 고치지 마세요.
# 실행하면 셀 실행 기록이 켜져, 채점 셀이 바로 위 셀의 출력을 읽습니다.
# 자동 생성 파일: 저장소에서 python -m z2h_check.bundle --write 로 다시 만듭니다 (원본: z2h_check/ 폴더).
# 원본 해시 sha256:24be8d6023b66a1ea287a13cd96bc63441070727a40e7ac3b2b5c1a0c1bd9345
# 출처 표기: 채점 기준의 강의 예제 숫자는 karpathy/nn-zero-to-hero, Value 동작 기준은 karpathy/micrograd (MIT).


def _z2h_install():
    import importlib.abc
    import importlib.util
    import sys

    sources = {
        'z2h_check': (
            '"""Z2H 노트북 채점 패키지.\n'
            '\n'
            '    from z2h_check import check, finish\n'
            '    check("L01-S02", globals())  # ✓ 맞음: … / ✕ 다시: … / ○ 아직: …\n'
            '    finish("L01")                # 채점표 + 결과코드 (+ 로컬 진도 서버로 보내기 시도)\n'
            '\n'
            '런타임 의존은 파이썬 표준 라이브러리뿐이다. IPython(Jupyter, Colab) 안에서 불러오면 셀 실행 기록(runs)을 켠다.\n'
            '호출 규약은 z2h_check/README.md 를 본다.\n'
            '"""\n'
            '__version__ = "0.2.0"\n'
            '\n'
            'from . import runs\n'
            'from .core import DEFAULT_SERVER_URL, check, finish, reset, results\n'
            'from .result_code import ResultCodeError, decode_result_code, encode_result_code\n'
            '\n'
            "runs.watch()  # 채점 셀이 '바로 위 셀이 보인 것'을 읽을 수 있게(IPython 이 아니면 아무것도 하지 않음)\n"
            '\n'
            '__all__ = [\n'
            '    "check", "finish", "results", "reset",\n'
            '    "encode_result_code", "decode_result_code", "ResultCodeError",\n'
            '    "DEFAULT_SERVER_URL", "__version__",\n'
            ']\n'
        ),
        'z2h_check._resources': (
            '"""패키지 안 자료 파일(기준값 JSON) 읽기.\n'
            '\n'
            '묶음(_bundle.py)으로 실행될 때는 파일이 없으므로, 묶음이 _EMBEDDED 에 넣어 둔 글을 쓴다.\n'
            '"""\n'
            'import json\n'
            'import os\n'
            '\n'
            '_EMBEDDED = {}  # 묶음이 채운다: {"reference/l01.json": "..."}\n'
            '_CACHE = {}\n'
            '\n'
            '\n'
            'def read_text(relpath):\n'
            '    if relpath in _EMBEDDED:\n'
            '        return _EMBEDDED[relpath]\n'
            '    path = os.path.join(os.path.dirname(os.path.abspath(__file__)), *relpath.split("/"))\n'
            '    with open(path, encoding="utf-8") as fh:\n'
            '        return fh.read()\n'
            '\n'
            '\n'
            'def load_json(relpath):\n'
            '    if relpath not in _CACHE:\n'
            '        _CACHE[relpath] = json.loads(read_text(relpath))\n'
            '    return _CACHE[relpath]\n'
        ),
        'z2h_check.core': (
            '"""check()와 finish(): 채점하고, 결과를 출력하고, 모듈 안에 기록한다.\n'
            '\n'
            '- check() 는 어떤 경우에도 예외를 밖으로 내보내지 않는다(Ctrl+C로 멈추는 것만 그대로 둔다).\n'
            '- 학생 코드가 틀리거나 비어 있으면 "✕ 다시", 채점 프로그램 자체가 고장 나면 "⚠ 채점기 오류"로 따로 보인다.\n'
            '- 아직 하지 않은 칸(필요한 셀을 아직 실행하지 않음, 빈 셀 그대로)은 "○ 아직"만 보이고 기록하지 않는다(이전 기록 그대로).\n'
            "- check()와 finish()를 부른 셀은 셀 실행 기록(runs)에 '도구 셀'로 표시한다. 채점 셀은 늘 바로 위 '내 셀'을 본다.\n"
            '- finish() 는 채점표와 결과코드를 늘 출력하고, 로컬 진도 서버(127.0.0.1)로 보내기를 시도한다.\n'
            '"""\n'
            'import contextlib\n'
            'import inspect\n'
            'import io\n'
            'import json\n'
            'import os\n'
            'import random\n'
            'import urllib.error\n'
            'import urllib.parse\n'
            'import urllib.request\n'
            'import warnings\n'
            '\n'
            'from . import graders, runs\n'
            'from .graders import _util\n'
            'from .result_code import encode_result_code, now_kst_iso\n'
            '\n'
            'DEFAULT_SERVER_URL = "http://127.0.0.1:8787/api/grade"\n'
            'SEND_TIMEOUT_S = 2.0\n'
            '_LOCAL_HOSTS = {"127.0.0.1", "localhost", "::1"}\n'
            '_PKG_DIR = os.path.dirname(os.path.abspath(__file__))\n'
            '_GRADER_SEED = 20221116  # 채점 안에서 쓰는 무작위 값을 고정한다(학생의 무작위 상태는 채점 뒤 되돌린다)\n'
            '\n'
            'MARK = {"pass": "✓ 맞음", "fail": "✕ 다시", "skip": "· 건너뜀", "error": "⚠ 채점기 오류"}\n'
            '_ROW_MARK = {"pass": "✓", "fail": "✕", "skip": "·", "error": "⚠", "todo": "○"}\n'
            '\n'
            '_RESULTS = {}  # check_id -> {"check_id", "status", "passed", "message_ko", "at", "attempts"}\n'
            '\n'
            '\n'
            'def reset():\n'
            '    """기록을 모두 지운다."""\n'
            '    _RESULTS.clear()\n'
            '\n'
            '\n'
            'def results():\n'
            '    """지금까지의 채점 기록 사본. status: pass | fail | skip | error"""\n'
            '    return {cid: dict(rec) for cid, rec in _RESULTS.items()}\n'
            '\n'
            '\n'
            'def _emit(text):\n'
            '    print(text)\n'
            '\n'
            '\n'
            'def _say(text):\n'
            '    try:\n'
            '        _emit(text)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException:\n'
            '        pass\n'
            '\n'
            '\n'
            'def _short(x):\n'
            '    return repr(x) if isinstance(x, str) else _util.type_name(x)\n'
            '\n'
            '\n'
            '# ---- check ------------------------------------------------------------------\n'
            '\n'
            'def check(check_id, *args, **kwargs):\n'
            '    """check("L01-S02", globals()) -> True/False. 맞으면 "✓ 맞음: …", 다르면 "✕ 다시: …"를 출력한다."""\n'
            '    try:\n'
            '        runs.mark_tool_cell()\n'
            '        return _check(check_id, args, kwargs)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:  # 마지막 안전망\n'
            '        _say(f"{MARK[\'error\']}: 채점 중 예상하지 못한 문제가 생겼습니다({type(exc).__name__}). 내 코드 탓이 아닐 수 있습니다.")\n'
            '        return False\n'
            '\n'
            '\n'
            'def _check(check_id, args, kwargs):\n'
            '    cid = graders.canonical_id(check_id)\n'
            '    if cid is None:\n'
            '        _say(f"{MARK[\'error\']}: 채점 번호 {_short(check_id)}를 모릅니다. 노트북의 채점 셀을 확인해 주세요. "\n'
            '             "(이 칸은 기록하지 않았습니다)")\n'
            '        return False\n'
            '    status, message, mark = _run_grader(cid, args, kwargs)\n'
            '    if status == "notyet":\n'
            '        _say(f"{mark}: {message} [{cid}]")  # 기록하지 않는다: 결과코드·서버 보내기·시도 횟수에 들어가지 않음\n'
            '        return False\n'
            '    prev = _RESULTS.get(cid)\n'
            '    _RESULTS[cid] = {\n'
            '        "check_id": cid,\n'
            '        "status": status,\n'
            '        "passed": status == "pass",\n'
            '        "mark": mark,\n'
            '        "message_ko": message,\n'
            '        "at": now_kst_iso(),\n'
            '        "attempts": (prev["attempts"] + 1) if prev else 1,\n'
            '    }\n'
            '    _say(f"{mark}: {message} [{cid}]")\n'
            '    return status == "pass"\n'
            '\n'
            '\n'
            '@contextlib.contextmanager\n'
            'def _quiet():\n'
            '    """채점하는 동안 학생 코드의 print·경고를 숨긴다."""\n'
            '    with contextlib.ExitStack() as stack:\n'
            '        stack.enter_context(contextlib.redirect_stdout(io.StringIO()))\n'
            '        stack.enter_context(contextlib.redirect_stderr(io.StringIO()))\n'
            '        stack.enter_context(warnings.catch_warnings())\n'
            '        warnings.simplefilter("ignore")\n'
            '        yield\n'
            '\n'
            '\n'
            'def _is_ours(filename):\n'
            '    return filename.startswith(_PKG_DIR + os.sep) or filename.startswith("<z2h_check")\n'
            '\n'
            '\n'
            'def _raised_in_package(exc):\n'
            '    tb = exc.__traceback__\n'
            '    last = None\n'
            '    while tb is not None:\n'
            '        last = tb\n'
            '        tb = tb.tb_next\n'
            '    return last is None or _is_ours(last.tb_frame.f_code.co_filename)\n'
            '\n'
            '\n'
            'def _run_grader(cid, args, kwargs):\n'
            '    """(status, 메시지, 화면 앞머리). status: pass | fail | skip | error | notyet"""\n'
            '    fn = graders.get_grader(cid)\n'
            '    try:\n'
            '        bound = inspect.signature(fn).bind(*args, **kwargs)\n'
            '    except TypeError:\n'
            '        return "fail", f"채점 칸에 넘긴 값의 개수나 이름이 다릅니다. 이렇게 부르세요: {graders.usage_of(cid)}", MARK["fail"]\n'
            '    state = random.getstate()\n'
            '    try:\n'
            '        random.seed(_GRADER_SEED)\n'
            '        with _quiet():\n'
            '            result = fn(*bound.args, **bound.kwargs)\n'
            '        if isinstance(result, _util.Pass):\n'
            '            return "pass", result.message_ko, result.mark\n'
            '        return "pass", str(result), MARK["pass"]\n'
            '    except _util.Fail as exc:\n'
            '        return "fail", exc.message_ko, MARK["fail"]\n'
            '    except _util.Skip as exc:\n'
            '        return "skip", exc.message_ko, MARK["skip"]\n'
            '    except _util.NotYet as exc:\n'
            '        return "notyet", exc.message_ko, exc.mark\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        if _raised_in_package(exc):\n'
            '            return "error", (f"채점 프로그램 안에서 문제가 생겼습니다({_util.describe_exc(exc)}). "\n'
            '                             "내 코드 탓이 아닐 수 있으니 이 칸은 넘어가고 알려 주세요."), MARK["error"]\n'
            '        return "fail", _util.student_error_message("내 코드를 실행", exc), MARK["fail"]\n'
            '    finally:\n'
            '        random.setstate(state)\n'
            '\n'
            '\n'
            '# ---- finish -----------------------------------------------------------------\n'
            '\n'
            'def finish(lecture_id="L01", send=True, server_url=None, timeout=SEND_TIMEOUT_S, at=None):\n'
            '    """이 편의 채점표를 출력하고 결과코드를 돌려준다. send=True면 로컬 진도 서버로도 보내 본다."""\n'
            '    try:\n'
            '        runs.mark_tool_cell()\n'
            '        return _finish(lecture_id, send, server_url, timeout, at)\n'
            '    except KeyboardInterrupt:\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        _say(f"{MARK[\'error\']}: 채점표를 만들다 문제가 생겼습니다({type(exc).__name__}).")\n'
            '        return ""\n'
            '\n'
            '\n'
            'def _finish(lecture_id, send, server_url, timeout, at):\n'
            '    lid = graders.canonical_lecture(lecture_id)\n'
            '    if lid is None:\n'
            '        _say(f"{MARK[\'error\']}: 편 번호 {_short(lecture_id)}를 모릅니다. finish(\\"L01\\") 처럼 부르세요.")\n'
            '        return ""\n'
            '    items = graders.lecture_checks(lid)\n'
            '    at = at or now_kst_iso()\n'
            '    counts = {"pass": 0, "fail": 0, "skip": 0, "error": 0, "todo": 0}\n'
            '    passed, failed, rows = [], [], []\n'
            '    width = max(len(cid) for cid, _ in items)\n'
            '    for cid, title in items:\n'
            '        rec = _RESULTS.get(cid)\n'
            '        status = rec["status"] if rec else "todo"\n'
            '        counts[status] += 1\n'
            '        if status == "pass":\n'
            '            passed.append(cid)\n'
            '        elif status == "fail":\n'
            '            failed.append(cid)\n'
            '        tail = {"pass": "",\n'
            '                "fail": f": {rec[\'message_ko\']}" if rec else "",\n'
            '                "skip": f": 건너뜀 ({rec[\'message_ko\']})" if rec else "",\n'
            '                "error": ": 채점기 오류",\n'
            '                "todo": ": 아직 안 함"}[status]\n'
            '        rows.append(f"  {_ROW_MARK[status]} {cid.ljust(width)}  {title}{tail}")\n'
            '    head = (f"{lid} 채점표 (모두 {len(items)}칸): 통과 {counts[\'pass\']} · 다시 {counts[\'fail\']} · "\n'
            '            f"건너뜀 {counts[\'skip\']} · 아직 안 함 {counts[\'todo\']}")\n'
            '    if counts["error"]:\n'
            '        head += f" · 채점기 오류 {counts[\'error\']}"\n'
            '    _say("\\n".join([head] + rows + [graders.mastery_note(lid)]))\n'
            '    code = encode_result_code(lid, passed, failed, at)\n'
            '    if send and os.environ.get("Z2H_NO_SEND", "") in ("", "0"):\n'
            '        _say(_send(lid, items, server_url, timeout, at))\n'
            '    _say("결과코드 (앱의 \'결과코드 붙여넣기\' 칸에 넣으세요):")\n'
            '    _say(code)\n'
            '    return code\n'
            '\n'
            '\n'
            'def _send(lid, items, server_url, timeout, at):\n'
            '    unreachable = "서버에 닿지 못했습니다. 아래 결과코드를 앱에 붙여 넣으세요. (Colab에서는 이것이 정상입니다)"\n'
            '    url = server_url or os.environ.get("Z2H_SERVER_URL") or DEFAULT_SERVER_URL\n'
            '    try:\n'
            '        parts = urllib.parse.urlsplit(url)\n'
            '        host = parts.hostname\n'
            '    except ValueError:\n'
            '        parts, host = None, None\n'
            '    if parts is None or parts.scheme != "http" or host not in _LOCAL_HOSTS:\n'
            '        return "진도 서버 주소는 http://127.0.0.1 만 씁니다. 보내지 않았습니다. 아래 결과코드를 앱에 붙여 넣으세요."\n'
            '    results_body = []\n'
            '    for cid, _title in items:\n'
            '        rec = _RESULTS.get(cid)\n'
            '        if rec and rec["status"] in ("pass", "fail"):\n'
            '            results_body.append({"check_id": cid, "passed": rec["status"] == "pass",\n'
            '                                 "message_ko": f"{rec[\'mark\']}: {rec[\'message_ko\']}"})\n'
            '    body = {"lecture_id": lid, "results": results_body, "at": at}\n'
            '    try:\n'
            '        status = _post_json(url, body, timeout)\n'
            '    except Exception:\n'
            '        return unreachable\n'
            '    if 200 <= status < 300:\n'
            '        return f"진도 서버에 채점 결과 {len(results_body)}칸을 보냈습니다. 앱을 새로 고치면 보입니다."\n'
            '    return f"서버가 결과를 받지 않았습니다(HTTP {status}). 아래 결과코드를 앱에 붙여 넣으세요."\n'
            '\n'
            '\n'
            'def _post_json(url, body, timeout):\n'
            '    """POST JSON. 프록시를 쓰지 않는다(127.0.0.1 전용). 돌려주는 값: HTTP 상태 번호."""\n'
            '    data = json.dumps(body, ensure_ascii=False).encode("utf-8")\n'
            '    parts = urllib.parse.urlsplit(url)\n'
            '    headers = {\n'
            '        "Content-Type": "application/json; charset=utf-8",\n'
            '        "Origin": f"{parts.scheme}://{parts.netloc}",\n'
            '        "User-Agent": "z2h-check/0.1",\n'
            '    }\n'
            '    req = urllib.request.Request(url, data=data, method="POST", headers=headers)\n'
            '    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))\n'
            '    try:\n'
            '        with opener.open(req, timeout=timeout) as resp:\n'
            '            return resp.status\n'
            '    except urllib.error.HTTPError as exc:\n'
            '        return exc.code\n'
        ),
        'z2h_check.graders': (
            '"""편별 채점 함수 모음. 지금은 1편(L01)만 있다."""\n'
            'from . import l01\n'
            '\n'
            '_LECTURES = {"L01": l01}\n'
            '\n'
            '\n'
            'def canonical_lecture(lecture_id):\n'
            '    """\'l01\', \' L01 \' -> \'L01\'. 모르는 편이면 None."""\n'
            '    if not isinstance(lecture_id, str):\n'
            '        return None\n'
            '    key = lecture_id.strip().upper()\n'
            '    return key if key in _LECTURES else None\n'
            '\n'
            '\n'
            'def lecture_checks(lecture_id):\n'
            '    """[(check_id, 제목)] 정본 순서. 모르는 편이면 []."""\n'
            '    lid = canonical_lecture(lecture_id)\n'
            '    return list(_LECTURES[lid].CHECKS) if lid else []\n'
            '\n'
            '\n'
            'def canonical_id(check_id):\n'
            '    """대소문자를 봐주고 정식 check_id 를 돌려준다 (\'l01-s02\' -> \'L01-S02\'). 모르면 None."""\n'
            '    if not isinstance(check_id, str):\n'
            '        return None\n'
            '    key = check_id.strip().lower()\n'
            '    for mod in _LECTURES.values():\n'
            '        for cid, _title in mod.CHECKS:\n'
            '            if cid.lower() == key:\n'
            '                return cid\n'
            '    return None\n'
            '\n'
            '\n'
            'def _module_of(check_id):\n'
            '    return _LECTURES[check_id.split("-", 1)[0]]\n'
            '\n'
            '\n'
            'def get_grader(check_id):\n'
            '    return _module_of(check_id).GRADERS[check_id]\n'
            '\n'
            '\n'
            'def usage_of(check_id):\n'
            '    return _module_of(check_id).USAGE[check_id]\n'
            '\n'
            '\n'
            'def title_of(check_id):\n'
            '    return dict(_module_of(check_id).CHECKS)[check_id]\n'
            '\n'
            '\n'
            'def mastery_note(lecture_id):\n'
            '    """finish() 채점표 끝 줄: 그 편의 익힘 기준(정본 mastery_rule). 모르는 편이면 빈 글."""\n'
            '    lid = canonical_lecture(lecture_id)\n'
            '    return _LECTURES[lid].mastery_note() if lid else ""\n'
        ),
        'z2h_check.graders._util': (
            '"""채점 함수들이 같이 쓰는 도구.\n'
            '\n'
            '원칙\n'
            '- 학생 코드(넘긴 이름 공간의 함수·클래스·값)는 늘 student()로 감싸 부른다. 거기서 난 오류는 Fail("✕ 다시")이 된다.\n'
            '- 학생 값을 숫자로 바꿀 때는 as_number()를 쓴다. 이 도구들은 어떤 입력에도 Fail 말고 다른 예외를 내지 않는다.\n'
            '  그래서 이 밖에서 난 예외는 채점기 자체의 오류로 본다.\n'
            '"""\n'
            'import math\n'
            'import re\n'
            '\n'
            '\n'
            'class Fail(Exception):\n'
            '    """학생 결과가 기준과 다름 ("✕ 다시"). message_ko 는 무엇이 다른지."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'class Skip(Exception):\n'
            '    """채점할 수 없어 건너뜀 (예: 셀 출력을 읽을 수 없는 곳). 통과도 실패도 아니다."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'class Pass:\n'
            '    """통과를 돌려줄 때 화면 앞머리를 바꾸고 싶으면 쓴다(예: 그림 단계 "✓ 실행됨"). 글만 돌려주면 "✓ 맞음"."""\n'
            '\n'
            '    def __init__(self, message_ko, mark):\n'
            '        self.message_ko = message_ko\n'
            '        self.mark = mark\n'
            '\n'
            '\n'
            'class NotYet(Exception):\n'
            '    """아직 하지 않음 ("○ 아직"): 필요한 셀을 아직 실행하지 않았거나 빈 셀 그대로일 때. 기록하지 않는다."""\n'
            '\n'
            '    def __init__(self, message_ko, mark="○ 아직"):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '        self.mark = mark\n'
            '\n'
            '\n'
            '# ---- 오류 설명 -------------------------------------------------------------\n'
            '\n'
            'def _safe_str(exc):\n'
            '    try:\n'
            '        text = str(exc)\n'
            '    except Exception:\n'
            '        text = "(오류 내용을 읽지 못함)"\n'
            '    text = " ".join(text.split())\n'
            '    return text if len(text) <= 160 else text[:157] + "..."\n'
            '\n'
            '\n'
            'def describe_exc(exc):\n'
            '    name = type(exc).__name__\n'
            '    text = _safe_str(exc)\n'
            '    return f"{name}: {text}" if text else name\n'
            '\n'
            '\n'
            '_WRAP_HINT = "숫자와 섞어 계산하려면 숫자를 Value로 감싸세요: other = other if isinstance(other, Value) else Value(other)"\n'
            '\n'
            '\n'
            'def hint_for(exc):\n'
            '    """자주 막히는 오류에 붙이는 한 줄 도움말 (정본 L01 막힘 힌트: __radd__, __rmul__, 감싸기, 셀 다시 실행)."""\n'
            '    text = _safe_str(exc)\n'
            '    if isinstance(exc, NameError) and "___" in text:\n'
            '        return "빈칸(___)이 아직 남아 있습니다. 빈칸을 채우고 그 셀부터 다시 실행하세요."\n'
            '    if isinstance(exc, NameError):\n'
            '        return "아직 정의하지 않은 이름이 있습니다. 앞 셀들을 위에서부터 다시 실행했나요?"\n'
            '    if isinstance(exc, SyntaxError):\n'
            '        return "코드 모양(괄호, 쌍점, 들여쓰기)이 맞지 않습니다. 오류가 가리키는 줄을 보세요."\n'
            '    if isinstance(exc, RecursionError):\n'
            '        return "같은 값을 끝없이 따라갑니다. 위상 정렬에서 visited로 이미 본 값을 건너뛰나요?"\n'
            '    if isinstance(exc, ZeroDivisionError):\n'
            '        return "0으로 나눴습니다."\n'
            '    if isinstance(exc, OverflowError):\n'
            '        return "숫자가 너무 커졌습니다."\n'
            '    if isinstance(exc, TypeError) and "unsupported operand" in text:\n'
            '        m = re.search(r"for (\\S+): \'(\\w+)\' and \'(\\w+)\'", text)\n'
            '        if m:\n'
            '            op, left, right = m.groups()\n'
            '            if left in ("int", "float") and right not in ("int", "float"):\n'
            '                if op == "+":\n'
            '                    return "숫자 + Value 를 하려면 __radd__가 필요합니다(sum()은 0부터 더합니다)."\n'
            '                name = {"*": "__rmul__", "-": "__rsub__", "/": "__rtruediv__"}.get(op)\n'
            '                if name:\n'
            '                    return f"숫자 {op} Value 를 하려면 {name}가 필요합니다."\n'
            '            if right in ("int", "float"):\n'
            '                return _WRAP_HINT\n'
            '    if isinstance(exc, AttributeError):\n'
            '        if re.search(r"\'(int|float)\' object has no attribute \'(data|grad|_prev)\'", text):\n'
            '            return _WRAP_HINT\n'
            '        m = re.search(r"has no attribute \'(\\w+)\'", text)\n'
            '        if m:\n'
            '            return f"{m.group(1)} 이(가) 아직 없습니다. 그 단계의 코드를 넣고 그 셀을 다시 실행했나요?"\n'
            '    return ""\n'
            '\n'
            '\n'
            'def student_error_message(what, exc):\n'
            '    hint = hint_for(exc)\n'
            '    msg = f"{what} 중 오류가 났습니다: {describe_exc(exc)}."\n'
            '    return f"{msg} {hint}" if hint else msg\n'
            '\n'
            '\n'
            'def student(what, fn, *args, **kwargs):\n'
            '    """학생 코드를 부른다. 오류가 나면 Fail(무엇을 하다 무슨 오류가 났는지)로 바꾼다."""\n'
            '    try:\n'
            '        return fn(*args, **kwargs)\n'
            '    except (KeyboardInterrupt, Fail, Skip, NotYet):\n'
            '        raise\n'
            '    except BaseException as exc:\n'
            '        raise Fail(student_error_message(what, exc)) from None\n'
            '\n'
            '\n'
            '# ---- 값 읽기 ---------------------------------------------------------------\n'
            '\n'
            'def type_name(x):\n'
            '    try:\n'
            '        return type(x).__name__\n'
            '    except Exception:\n'
            '        return "알 수 없는 것"\n'
            '\n'
            '\n'
            'def fmt(x):\n'
            '    """화면용 숫자: 2.0, -3.0, 14.003, 0.707107"""\n'
            '    if isinstance(x, bool):\n'
            '        return str(x)\n'
            '    if isinstance(x, int):\n'
            '        return str(x)\n'
            '    if isinstance(x, float):\n'
            '        if math.isfinite(x) and x == int(x) and abs(x) < 1e15:\n'
            '            return f"{x:.1f}"\n'
            '        return f"{x:.6g}"\n'
            '    return str(x)\n'
            '\n'
            '\n'
            'def as_number(x, label, _depth=0):\n'
            '    """학생 값을 float로. 텐서·numpy처럼 .item()이 있으면 그 값."""\n'
            '    if isinstance(x, bool) or x is None:\n'
            '        raise Fail(f"{label}에는 숫자가 와야 하는데 {x}가 왔습니다.")\n'
            '    if isinstance(x, (int, float)):\n'
            '        v = float(x)\n'
            '    elif isinstance(x, (str, bytes)) or _depth >= 3:\n'
            '        raise Fail(f"{label}에는 숫자가 와야 하는데 {type_name(x)}이(가) 왔습니다.")\n'
            '    else:\n'
            '        item = getattr(x, "item", None) if not isinstance(x, type) else None\n'
            '        if callable(item):\n'
            '            return as_number(student(f"{label}을(를) 숫자로 바꾸기", item), label, _depth + 1)\n'
            '        v = student(f"{label}을(를) 숫자로 바꾸기", float, x)\n'
            '    if not math.isfinite(v):\n'
            '        raise Fail(f"{label}: 내 값 {v}. 계산 중 숫자가 넘쳤거나 0으로 나눴습니다.")\n'
            '    return v\n'
            '\n'
            '\n'
            'def close(got, want, tol):\n'
            '    return abs(got - want) <= tol * max(1.0, abs(want))\n'
            '\n'
            '\n'
            '# ---- 출력 글 비교 ------------------------------------------------------------\n'
            '\n'
            '# 앞뒤가 글자·숫자·점에 붙지 않은 숫자만 숫자로 본다(x1 같은 이름, 0x13d20f5b0 같은 메모리 주소 안 숫자는 글자로 본다)\n'
            'NUMBER = re.compile(r"(?<![\\w.])-?\\d+(?:\\.\\d*)?(?:[eE][-+]?\\d+)?(?![\\w.])")\n'
            '\n'
            '\n'
            'def _pattern(expected):\n'
            '    """기대 출력 글을 정규식으로: 숫자 자리는 아무 숫자나 잡고(나중에 허용 오차로 비교), 빈칸은 빈칸 여러 개와도 맞는다."""\n'
            '    parts, numbers, pos = [], [], 0\n'
            '    for m in NUMBER.finditer(expected):\n'
            '        parts.append(_text_pattern(expected[pos:m.start()]))\n'
            '        parts.append("(" + NUMBER.pattern + ")")\n'
            '        numbers.append(float(m.group()))\n'
            '        pos = m.end()\n'
            '    parts.append(_text_pattern(expected[pos:]))\n'
            '    return re.compile("".join(parts)), numbers\n'
            '\n'
            '\n'
            'def _text_pattern(text):\n'
            '    return "".join(r"\\s+" if piece.isspace() else re.escape(piece) for piece in re.split(r"(\\s+)", text) if piece)\n'
            '\n'
            '\n'
            'def output_has(text, expected, tol):\n'
            '    """출력 글 text 안에 기대 출력 expected 가 있는지(정본 규칙: contains, 소수는 허용 오차 tol)."""\n'
            '    pattern, numbers = _pattern(expected.strip())\n'
            '    pos = 0\n'
            '    while True:\n'
            '        m = pattern.search(text, pos)\n'
            '        if m is None:\n'
            '            return False\n'
            '        if all(close(float(got), want, tol) for got, want in zip(m.groups(), numbers)):\n'
            '            return True\n'
            '        pos = m.start() + 1\n'
            '\n'
            '\n'
            'def last_number(text):\n'
            '    found = NUMBER.findall(text)\n'
            '    return float(found[-1]) if found else None\n'
            '\n'
            '\n'
            'def preview(text, limit=80):\n'
            '    """학생 출력을 한 줄로 짧게: 줄바꿈은 \' / \' 로."""\n'
            '    flat = " / ".join(line.strip() for line in text.strip().splitlines() if line.strip())\n'
            '    return flat if len(flat) <= limit else flat[:limit - 3] + "..."\n'
        ),
        'z2h_check.graders.l01': (
            '# 출처: 채점 기준(정본 expected_check, 재구성 check_code)의 강의 예제 숫자는 karpathy/nn-zero-to-hero (lectures/micrograd), Value 동작 기준은 karpathy/micrograd (engine.py), 둘 다 MIT License. 채점 코드는 새로 씀.\n'
            '"""1편 micrograd 채점: 기준값 파일(reference/l01.json, 정본 L01.json 에서 z2h_check.refgen 이 만든 것)을 그대로 채점 칸으로 쓴다.\n'
            '\n'
            '채점 칸 = 정본의 코드 단계(정본 순서) + 재구성 조각. 모두 check("<칸 id>", globals()) 로 부른다.\n'
            '단계 칸은 정본 expected_check 의 모양대로 본다.\n'
            '- value_range + expr: 식을 학생 이름 공간(사본)에서 계산해 min~max 안인지. 식이 읽는 이름(기준값 reads)이 아직 없으면 "○ 아직".\n'
            "- output_match: 바로 위 '내 셀'이 보인 글(print 출력 + 마지막 줄 값)에 기대 출력이 들어 있는지(소수는 허용 오차).\n"
            '- value_range(식 없음): 바로 위 셀 출력의 마지막 숫자가 min~max 안인지.\n'
            '- manual: 바로 위 셀이 오류 없이 돌았는지만 본다. 통과로 기록하되 화면에는 "✓ 실행됨"으로 보이고,\n'
            "  그림의 숫자는 학습자가 '스스로 확인할 것' 목록으로 확인한다.\n"
            '재구성 칸은 정본 확인 셀(check_code)을 학생 이름 공간의 사본에서 실행하고, 그 출력을 통과 기준(pass_ko)대로 본다.\n'
            '"""\n'
            'import builtins\n'
            'import contextlib\n'
            'import io\n'
            '\n'
            'from . import _util as U\n'
            'from .. import runs\n'
            'from .._resources import load_json\n'
            '\n'
            'LECTURE_ID = "L01"\n'
            'MANUAL_MARK = "✓ 실행됨"  # 그림 단계: 통과(p)로 기록하지만 내용은 학습자가 스스로 확인하므로 \'맞음\'이라 하지 않는다\n'
            '\n'
            '\n'
            'def ref():\n'
            '    return load_json("reference/l01.json")\n'
            '\n'
            '\n'
            'def tol():\n'
            '    return ref()["tolerance"]["number"]\n'
            '\n'
            '\n'
            'def usage(check_id):\n'
            '    return f\'check("{check_id}", globals())\'\n'
            '\n'
            '\n'
            'F = U.fmt\n'
            '\n'
            '\n'
            '# ---- 이름 공간 다루기 -----------------------------------------------------------\n'
            '\n'
            'def _need_ns(ns, check_id):\n'
            '    if not isinstance(ns, dict):\n'
            '        raise U.Fail(f"채점 셀을 고치지 말고 그대로 실행하세요: {usage(check_id)}")\n'
            '    return ns\n'
            '\n'
            '\n'
            'def _missing(names, ns):\n'
            '    return sorted(name for name in names if name not in ns and not hasattr(builtins, name))\n'
            '\n'
            '\n'
            'def _not_yet_names(missing, where):\n'
            '    shown = ", ".join(missing[:4])\n'
            '    return U.NotYet(f"{where}에 필요한 이름 {shown}이(가) 아직 없어요. 위 셀을 먼저 실행한 뒤 이 채점 셀을 다시 실행하세요.")\n'
            '\n'
            '\n'
            'def _eval_number(expr, reads, ns):\n'
            '    missing = _missing(reads, ns)\n'
            '    if missing:\n'
            '        raise _not_yet_names(missing, f"확인 식 {expr}")\n'
            '    code = compile(expr, "<확인 식>", "eval")\n'
            '    value = U.student(f"확인 식 {expr} 계산", eval, code, dict(ns))  # 정본 확인 식을 학생 이름 공간의 사본에서 계산\n'
            '    return U.as_number(value, f"확인 식 {expr}")\n'
            '\n'
            '\n'
            '# ---- 바로 위 셀 -------------------------------------------------------------------\n'
            '\n'
            'def _last_cell():\n'
            '    cell = runs.last_cell()\n'
            '    if cell is None:\n'
            '        if runs.watching():\n'
            '            raise U.NotYet("바로 위 단계 셀을 먼저 실행한 뒤 이 채점 셀을 다시 실행하세요.")\n'
            '        raise U.Skip("이 실행 환경에서는 셀 출력을 읽지 못해, 출력으로 보는 칸은 건너뜁니다.")\n'
            '    if cell["blank"]:\n'
            '        raise U.NotYet("위 빈 셀에 코드를 입력해 실행한 뒤 이 채점 셀을 다시 실행하세요.")\n'
            '    if cell["error"] is not None:\n'
            '        raise U.Fail(U.student_error_message("바로 위 셀 실행", cell["error"]))\n'
            '    return cell\n'
            '\n'
            '\n'
            'def _shown_text(cell):\n'
            '    return "\\n".join(part for part in (cell["stdout"], cell["shown"]) if part)\n'
            '\n'
            '\n'
            'def _range_text(exp):\n'
            '    lo, hi = exp["min"], exp["max"]\n'
            '    return F(lo) if lo == hi else f"{F(lo)}~{F(hi)}"\n'
            '\n'
            '\n'
            'def _value_text(value, exp):\n'
            '    """정본 범위가 정수(개수)이면 내 값도 정수로 보여 준다."""\n'
            '    if isinstance(exp["min"], int) and isinstance(exp["max"], int) and value.is_integer():\n'
            '        return str(int(value))\n'
            '    return F(value)\n'
            '\n'
            '\n'
            'def _in_range(value, exp, label):\n'
            '    shown = _value_text(value, exp)\n'
            '    if exp["min"] <= value <= exp["max"]:\n'
            '        return f"{label} = {shown} (기준 {_range_text(exp)})."\n'
            '    raise U.Fail(f"{label} = {shown}. 기준은 {_range_text(exp)} 입니다. 앱의 \'보여야 하는 것\'과 비교해 보세요.")\n'
            '\n'
            '\n'
            '# ---- 단계 칸 -------------------------------------------------------------------\n'
            '\n'
            'def _grade_step(spec, ns):\n'
            '    exp = spec["expected_check"]\n'
            '    kind = exp["type"]\n'
            '    if kind == "value_range" and exp.get("expr"):\n'
            '        value = _eval_number(exp["expr"], spec["reads"], _need_ns(ns, spec["step_id"]))\n'
            '        return _in_range(value, exp, f"확인 식 {exp[\'expr\']}")\n'
            '    cell = _last_cell()\n'
            '    text = _shown_text(cell)\n'
            '    if kind == "output_match":\n'
            '        want = exp["expected"]\n'
            '        if U.output_has(text, want, tol()):\n'
            '            return f"출력에 {U.preview(want)} 이(가) 보입니다."\n'
            '        got = U.preview(text) or "(출력 없음)"\n'
            '        raise U.Fail(f"출력에서 {U.preview(want)} 을(를) 찾지 못했습니다(소수는 {tol():g}까지 같으면 맞음). "\n'
            '                     f"내 출력: {got}. 앱의 \'보여야 하는 것\'과 비교해 보세요.")\n'
            '    if kind == "value_range":\n'
            '        value = U.last_number(text)\n'
            '        if value is None:\n'
            '            raise U.Fail(f"바로 위 셀 출력에 숫자가 없습니다. {_range_text(exp)} 사이 숫자가 찍혀야 합니다.")\n'
            '        return _in_range(value, exp, "출력의 마지막 숫자")\n'
            '    if kind == "manual":\n'
            '        return U.Pass("셀이 오류 없이 돌았습니다. 그림의 숫자는 위 \'보여야 하는 것\'의 \'스스로 확인할 것\'으로 직접 확인하세요.",\n'
            '                      mark=MANUAL_MARK)\n'
            '    raise ValueError(f"모르는 expected_check.type: {kind}")\n'
            '\n'
            '\n'
            'def _step_grader(spec):\n'
            '    def grade(ns=None):\n'
            '        return _grade_step(spec, ns)\n'
            '    return grade\n'
            '\n'
            '\n'
            '# ---- 재구성 칸 -----------------------------------------------------------------\n'
            '\n'
            'def _run_check_code(rebuild_id, code, ns):\n'
            '    """정본 확인 셀을 학생 이름 공간의 사본에서 실행하고 찍힌 글을 돌려준다(학생 이름은 바꾸지 않는다)."""\n'
            '    out = io.StringIO()\n'
            '    with contextlib.redirect_stdout(out):\n'
            '        U.student("확인 셀 실행", exec, compile(code, f"<{rebuild_id} 확인 셀>", "exec"), dict(ns))\n'
            '    return out.getvalue()\n'
            '\n'
            '\n'
            'def _grade_rebuild(spec, ns):\n'
            '    rid = spec["rebuild_id"]\n'
            '    ns = _need_ns(ns, rid)\n'
            '    missing = _missing(spec["reads"], ns)\n'
            '    if missing:\n'
            '        raise _not_yet_names(missing, "확인 셀")\n'
            '    text = _run_check_code(rid, spec["check_code"], ns)\n'
            '    got = U.preview(text) or "(출력 없음)"\n'
            '    rule = spec["pass"]\n'
            '    if rule["rule"] == "output_match":\n'
            '        if U.output_has(text, rule["expected"], tol()):\n'
            '            return f"확인 셀 출력 {got} 이(가) 통과 기준과 같습니다."\n'
            '        raise U.Fail(f"확인 셀 출력이 {got} 입니다. {U.preview(rule[\'expected\'])} 이(가) 나와야 통과입니다.")\n'
            '    if rule["rule"] == "ends_with":\n'
            '        if text.split() and text.split()[-1] == rule["text"]:\n'
            '            return f"확인 셀 출력 {got} 의 끝이 {rule[\'text\']} 입니다."\n'
            '        raise U.Fail(f"확인 셀 출력이 {got} 입니다. 끝이 {rule[\'text\']} 여야 통과입니다.")\n'
            '    raise ValueError(f"모르는 통과 기준: {rule[\'rule\']}")\n'
            '\n'
            '\n'
            'def _rebuild_grader(spec):\n'
            '    def grade(ns=None):\n'
            '        return _grade_rebuild(spec, ns)\n'
            '    return grade\n'
            '\n'
            '\n'
            '# ---- 등록 ----------------------------------------------------------------------\n'
            '\n'
            'def _register():\n'
            '    checks, graders, usages = [], {}, {}\n'
            '    for spec in ref()["steps"]:\n'
            '        cid = spec["step_id"]\n'
            '        checks.append((cid, spec["title_ko"]))\n'
            '        graders[cid] = _step_grader(spec)\n'
            '        usages[cid] = usage(cid)\n'
            '    for spec in ref()["rebuild"]:\n'
            '        rid = spec["rebuild_id"]\n'
            '        checks.append((rid, spec["title_ko"]))\n'
            '        graders[rid] = _rebuild_grader(spec)\n'
            '        usages[rid] = usage(rid)\n'
            '    return checks, graders, usages\n'
            '\n'
            '\n'
            'CHECKS, GRADERS, USAGE = _register()  # CHECKS: [(check_id, 제목)] 기준값 파일 순서(= 정본 순서)\n'
            '\n'
            '\n'
            'def mastery_note():\n'
            '    """finish() 채점표 끝에 붙이는 익힘 기준 한 줄(정본 mastery_rule)."""\n'
            '    rule = ref()["mastery_rule"]\n'
            '    return (f"익힘 기준(앱이 판정): 단계 완료 {rule[\'steps_complete_ratio\']:.0%}(선택 단계 제외) · "\n'
            '            f"확인 문제 {rule[\'question_score_ratio\']:.0%} 이상 · 재구성은 힌트 H{rule[\'rebuild_max_hint_level\']} 이하로 통과.")\n'
        ),
        'z2h_check.rebuild': (
            '"""재구성(빈 셀에서 다시 만들기) 구역의 셀 도우미.\n'
            '\n'
            '    import z2h_check.rebuild as _z2h_rebuild\n'
            "    _z2h_rebuild.start(globals())             # '재구성 시작' 셀: 이 노트북을 빈 노트북처럼\n"
            '    _z2h_rebuild.resume(globals(), "L01-R3")  # \'R3 시작\' 셀: 본문 상태로 되돌리기\n'
            '\n'
            '정본 rebuild_tasks: R1 은 빈 노트북에서 Value 를 다시 만들고, R2 는 R1 셀에 backward()를 덧붙이고,\n'
            "R3 은 '이 편의 노트북에서 Value, Neuron, Layer, MLP, 데이터 셀까지 실행된 상태로' 시작한다.\n"
            '- start(): 본문에서 만든 이름(밑줄로 시작하지 않는 것)을 따로 보관하고 지운다. 채점 도구와 채점 기록은 그대로.\n'
            '  두 번 눌러도 처음 보관한 본문을 지키고 지우기만 한다.\n'
            '- resume(): 보관한 본문 이름을 되돌리고, 그 조각의 확인 셀이 읽는 이름(기준값 파일 reads, R3 은 loss)을 지운다.\n'
            '  그래서 확인은 이 조각의 새 셀이 만든 값만 본다. 보관한 것이 없으면(다른 날 새 커널) 무엇을 먼저 실행할지 알려 준다.\n'
            '"""\n'
            'from . import runs\n'
            'from ._resources import load_json\n'
            '\n'
            'SAVED = "_z2h_lecture_names"\n'
            'KEEP = frozenset({"check", "finish", "In", "Out", "get_ipython", "exit", "quit"})\n'
            '\n'
            '\n'
            'def _kept_names():\n'
            '    keep = set(KEEP)\n'
            '    shell = runs._shell()\n'
            '    if shell is not None:\n'
            '        keep |= set(getattr(shell, "user_ns_hidden", {}) or {})  # IPython 이 스스로 넣어 둔 이름\n'
            '    return keep\n'
            '\n'
            '\n'
            'def _clearable(ns):\n'
            '    keep = _kept_names()\n'
            '    return sorted(name for name in ns if not name.startswith("_") and name not in keep)\n'
            '\n'
            '\n'
            'def _spec(rebuild_id):\n'
            '    for spec in load_json("reference/l01.json")["rebuild"]:\n'
            '        if spec["rebuild_id"] == rebuild_id:\n'
            '            return spec\n'
            '    raise KeyError(rebuild_id)\n'
            '\n'
            '\n'
            'def _names(names, limit=6):\n'
            '    return ", ".join(names[:limit]) + (" 등" if len(names) > limit else "")\n'
            '\n'
            '\n'
            'def start(ns):\n'
            '    """\'재구성 시작\' 셀: 본문 이름을 보관하고 지워 빈 노트북처럼 만든다."""\n'
            '    runs.mark_tool_cell()\n'
            '    gone = _clearable(ns)\n'
            '    if SAVED not in ns:\n'
            '        ns[SAVED] = {name: ns[name] for name in gone}\n'
            '    for name in gone:\n'
            '        del ns[name]\n'
            '    if gone:\n'
            '        print(f"재구성 시작: 위에서 만든 이름 {len(gone)}개({_names(gone)})를 따로 보관하고 지워 빈 노트북처럼 만들었습니다. "\n'
            '              "채점 기록은 그대로입니다.")\n'
            '    else:\n'
            '        print("재구성 시작: 지울 이름이 없습니다(빈 노트북 상태). 채점 기록은 그대로입니다.")\n'
            '    print("이제 아래 조각의 빈 셀에 처음부터 다시 만들어 보세요. 위 본문 셀은 다시 실행하지 않습니다.")\n'
            '\n'
            '\n'
            'def resume(ns, rebuild_id):\n'
            '    """\'R3 시작\' 같은 셀: 보관한 본문 이름을 되돌리고, 확인 셀이 읽는 이름은 지운다."""\n'
            '    runs.mark_tool_cell()\n'
            '    spec = _spec(rebuild_id)\n'
            '    saved = ns.get(SAVED) or {}\n'
            '    if saved:\n'
            '        for name in _clearable(ns):\n'
            '            del ns[name]\n'
            '        ns.update(saved)\n'
            '    dropped = [name for name in spec["reads"] if ns.pop(name, None) is not None]\n'
            '    if saved:\n'
            '        print(f"{rebuild_id} 시작: 보관해 둔 본문 이름 {len(saved)}개를 되돌렸습니다(재구성 시작 셀 전 상태).")\n'
            '    missing = [name for name in spec["needs"] if name not in ns]\n'
            '    if missing:\n'
            '        print(f"{rebuild_id} 시작: 본문 이름 {_names(missing)}이(가) 없습니다. 위 본문의 그 셀들을 먼저 실행한 뒤 "\n'
            '              "이 셀을 다시 실행하세요(다른 날이면 맨 위 준비 셀부터).")\n'
            '    if dropped:\n'
            '        print(f"확인 셀이 보는 {_names(dropped)}은(는) 지웠습니다. 아래 새 셀이 만든 값만 확인합니다.")\n'
        ),
        'z2h_check.result_code': (
            '"""결과코드: Z2H1.{base64url(JSON)}.{crc32 8자리}\n'
            '\n'
            '규칙 (진도 서버·앱의 해독기와 같다. 바꾸지 않는다):\n'
            '- JSON은 키 순서 v, l, p, f, t 고정, 공백 없음.\n'
            '  예: {"v":1,"l":"L01","p":["L01-S11b"],"f":["L01-R2"],"t":"2026-11-16T21:00:00+09:00"}\n'
            '  p = 통과한 check_id, f = 실패한 check_id (L01-S11b 같은 전체 문자열). 둘 다 없으면 "아직 안 함".\n'
            '- base64url은 패딩(=)을 뗀다.\n'
            '- crc32는 base64url 문자열의 UTF-8 바이트로 계산해 8자리 소문자 16진수로 쓴다.\n'
            '비밀값이 아니며 위조를 막지 않는다. 목적은 붙여넣기 오타 잡기다.\n'
            '"""\n'
            'import base64\n'
            'import binascii\n'
            'import datetime\n'
            'import json\n'
            'import re\n'
            'import zlib\n'
            '\n'
            'PREFIX = "Z2H1"\n'
            'VERSION = 1\n'
            'KEYS = ("v", "l", "p", "f", "t")\n'
            'KST = datetime.timezone(datetime.timedelta(hours=9))\n'
            '\n'
            '_B64 = re.compile(r"[A-Za-z0-9_-]+")\n'
            '_CRC = re.compile(r"[0-9a-f]{8}")\n'
            '\n'
            'MSG_SHAPE = "결과코드 모양이 아닙니다. 노트북 finish() 아래에 나온 \'Z2H1.\'로 시작하는 줄 전체를 복사해 주세요."\n'
            'MSG_CHANGED = "코드가 잘렸거나 글자가 바뀌었습니다. 노트북에서 결과코드를 다시 복사해 주세요."\n'
            'MSG_CONTENT = "결과코드 안의 내용을 읽지 못했습니다. 노트북에서 finish()를 다시 실행해 새 코드를 받아 주세요."\n'
            'MSG_VERSION = "이 결과코드는 다른 버전에서 만들었습니다. 노트북의 준비 셀을 새로 받아 다시 실행해 주세요."\n'
            '\n'
            '\n'
            'class ResultCodeError(ValueError):\n'
            '    """결과코드를 풀지 못함. message_ko 는 화면에 그대로 보여 줄 한국어 안내."""\n'
            '\n'
            '    def __init__(self, message_ko):\n'
            '        super().__init__(message_ko)\n'
            '        self.message_ko = message_ko\n'
            '\n'
            '\n'
            'def now_kst_iso():\n'
            '    """지금 서울 시각, 예: 2026-11-16T21:00:00+09:00"""\n'
            '    return datetime.datetime.now(KST).replace(microsecond=0).isoformat()\n'
            '\n'
            '\n'
            'def crc32_hex(text):\n'
            '    return format(zlib.crc32(text.encode("utf-8")) & 0xFFFFFFFF, "08x")\n'
            '\n'
            '\n'
            'def encode_result_code(lecture_id, passed, failed, at=None):\n'
            '    """encode_result_code("L01", ["L01-S11b"], ["L01-R2"], "2026-11-16T21:00:00+09:00") -> "Z2H1.....32dccd52"\n'
            '\n'
            '    passed·failed 는 넘긴 순서 그대로 담는다. at 이 없으면 지금 서울 시각.\n'
            '    """\n'
            '    payload = {\n'
            '        "v": VERSION,\n'
            '        "l": str(lecture_id),\n'
            '        "p": [str(x) for x in passed],\n'
            '        "f": [str(x) for x in failed],\n'
            '        "t": at or now_kst_iso(),\n'
            '    }\n'
            '    text = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))\n'
            '    b64 = base64.urlsafe_b64encode(text.encode("utf-8")).decode("ascii").rstrip("=")\n'
            '    return f"{PREFIX}.{b64}.{crc32_hex(b64)}"\n'
            '\n'
            '\n'
            'def decode_result_code(code):\n'
            '    """결과코드를 풀어 {"v":1,"l":...,"p":[...],"f":[...],"t":...} 를 돌려준다.\n'
            '\n'
            '    앞뒤·중간의 공백과 줄바꿈은 무시한다. 틀리면 ResultCodeError(한국어 안내)를 던진다.\n'
            '    """\n'
            '    if not isinstance(code, str):\n'
            '        raise ResultCodeError(MSG_SHAPE)\n'
            '    text = "".join(code.split())\n'
            '    parts = text.split(".")\n'
            '    if len(parts) != 3 or parts[0] != PREFIX:\n'
            '        raise ResultCodeError(MSG_SHAPE)\n'
            '    _prefix, b64, crc = parts\n'
            '    if not _B64.fullmatch(b64) or not _CRC.fullmatch(crc):\n'
            '        raise ResultCodeError(MSG_CHANGED)\n'
            '    if crc32_hex(b64) != crc:\n'
            '        raise ResultCodeError(MSG_CHANGED)\n'
            '    try:\n'
            '        raw = base64.urlsafe_b64decode(b64 + "=" * (-len(b64) % 4))\n'
            '        obj = json.loads(raw.decode("utf-8"))\n'
            '    except (binascii.Error, ValueError, UnicodeDecodeError):\n'
            '        raise ResultCodeError(MSG_CONTENT) from None\n'
            '    if not isinstance(obj, dict) or set(obj) != set(KEYS):\n'
            '        raise ResultCodeError(MSG_CONTENT)\n'
            '    if type(obj["v"]) is not int or obj["v"] != VERSION:\n'
            '        raise ResultCodeError(MSG_VERSION)\n'
            '    if not isinstance(obj["l"], str) or not isinstance(obj["t"], str):\n'
            '        raise ResultCodeError(MSG_CONTENT)\n'
            '    for key in ("p", "f"):\n'
            '        if not isinstance(obj[key], list) or not all(isinstance(x, str) for x in obj[key]):\n'
            '            raise ResultCodeError(MSG_CONTENT)\n'
            '    return {key: obj[key] for key in KEYS}\n'
        ),
        'z2h_check.runs': (
            '"""셀 실행 기록: 셀이 끝날 때마다 그 셀이 화면에 보인 것(출력 글, 마지막 줄 값, 오류)을 기억한다.\n'
            '\n'
            '    from z2h_check import runs\n'
            '    runs.watch()        # z2h_check 를 불러올 때 한 번 부른다(IPython 이 아니면 아무것도 걸지 않음)\n'
            "    runs.last_cell()    # 채점 셀이 부른다: 바로 앞에 실행한 '내 셀'의 기록, 없으면 None\n"
            '\n'
            '정본 expected_check 의 output_match(출력 글 대조), 식이 없는 value_range(출력의 마지막 숫자), manual(셀이 오류 없이\n'
            "돌았는지)는 '바로 위 셀이 보인 것'으로 채점한다. 그래서 IPython(Jupyter, Colab)의 셀 실행 앞뒤 알림\n"
            '(pre_run_cell, post_run_cell)에 작은 함수를 건다.\n'
            "- 셀 앞: sys.stdout 을 '받아 적으며 그대로 내보내는' 것으로 잠깐 바꾼다. 화면에 보이는 것은 그대로다.\n"
            '- 셀 뒤: sys.stdout 을 되돌리고 (셀 소스, 출력 글, 마지막 줄 값의 글, 오류, 빈 셀인지)를 기록한다.\n'
            '- 채점 도구를 부른 셀(check, finish, 준비 셀, 재구성 시작 셀)은 mark_tool_cell() 로 표시해 기록하지 않는다.\n'
            "  그래서 채점 셀을 두 번 눌러도 늘 바로 위 '내 셀'을 본다.\n"
            "- 주석과 빈 줄뿐인 셀(학생 노트북의 '# 여기에 작성')은 blank=True 로 기록한다.\n"
            '준비 셀(묶음)을 다시 실행해도 이미 걸린 함수를 찾아 그대로 쓴다(두 번 걸지 않는다).\n'
            'IPython 이 아니면 record_cell() 로 넣은 기록만 본다(시험과 일반 파이썬용).\n'
            '"""\n'
            'import sys\n'
            '\n'
            'EVENTS = ("pre_run_cell", "post_run_cell")\n'
            'STATE_VERSION = 2\n'
            'MAX_TEXT = 100000  # 셀 출력은 앞뒤 절반씩만 기억한다\n'
            '\n'
            '\n'
            'class _Tee:\n'
            '    """sys.stdout 대신 잠깐 두는 것: 받은 글을 적어 두고 원래 출력으로 그대로 보낸다."""\n'
            '\n'
            '    def __init__(self, target):\n'
            '        self._z2h_target = target\n'
            '        self._z2h_parts = []\n'
            '\n'
            '    def write(self, text):\n'
            '        if isinstance(text, str):\n'
            '            self._z2h_parts.append(text)\n'
            '        return self._z2h_target.write(text)\n'
            '\n'
            '    def text(self):\n'
            '        return "".join(self._z2h_parts)\n'
            '\n'
            '    def __getattr__(self, name):\n'
            '        if name.startswith("_z2h_"):\n'
            '            raise AttributeError(name)\n'
            '        return getattr(self._z2h_target, name)\n'
            '\n'
            '\n'
            'def _new_state():\n'
            '    return {"version": STATE_VERSION, "last": None, "stack": [], "tool": False}\n'
            '\n'
            '\n'
            '_FALLBACK = _new_state()\n'
            '\n'
            '\n'
            'def _shell():\n'
            '    """지금 IPython 안이면 그 셸, 아니면 None. IPython 을 새로 불러오지는 않는다."""\n'
            '    ipy = sys.modules.get("IPython")\n'
            '    if ipy is None:\n'
            '        return None\n'
            '    try:\n'
            '        shell = ipy.get_ipython()\n'
            '    except Exception:\n'
            '        return None\n'
            '    if shell is None or not hasattr(shell, "events"):\n'
            '        return None\n'
            '    return shell\n'
            '\n'
            '\n'
            'def _find_hooks(shell):\n'
            '    """이 셸에 걸린 우리 함수 (앞, 뒤). 버전이 같은 것만."""\n'
            '    found = []\n'
            '    for event in EVENTS:\n'
            '        try:\n'
            '            callbacks = list(shell.events.callbacks.get(event, []))\n'
            '        except Exception:\n'
            '            return None\n'
            '        hook = next((cb for cb in callbacks if getattr(cb, "z2h_runs_state", {}).get("version") == STATE_VERSION), None)\n'
            '        if hook is None:\n'
            '            return None\n'
            '        found.append(hook)\n'
            '    return tuple(found)\n'
            '\n'
            '\n'
            'def _state():\n'
            '    shell = _shell()\n'
            '    hooks = _find_hooks(shell) if shell is not None else None\n'
            '    return hooks[0].z2h_runs_state if hooks else _FALLBACK\n'
            '\n'
            '\n'
            'def _cap(text):\n'
            '    if len(text) <= MAX_TEXT:\n'
            '        return text\n'
            '    half = MAX_TEXT // 2\n'
            '    return text[:half] + "\\n...\\n" + text[-half:]\n'
            '\n'
            '\n'
            'def is_blank(source):\n'
            '    return all(not line.strip() or line.strip().startswith("#") for line in (source or "").splitlines())\n'
            '\n'
            '\n'
            'def _store(state, source, stdout, shown, error):\n'
            '    state["last"] = {"source": source or "", "stdout": _cap(stdout or ""), "shown": shown,\n'
            '                     "error": error, "blank": is_blank(source)}\n'
            '\n'
            '\n'
            'def record_cell(source, stdout="", shown=None, error=None):\n'
            '    """셀 하나의 기록을 넣는다(IPython 알림이 부르는 것과 같은 길. 시험과 일반 파이썬에서도 쓴다)."""\n'
            '    _store(_state(), source, stdout, None if shown is None else _cap(str(shown)), error)\n'
            '\n'
            '\n'
            'def mark_tool_cell():\n'
            '    """지금 셀은 채점 도구 셀이다: 이 셀이 끝나도 \'바로 위 내 셀\' 기록을 바꾸지 않는다."""\n'
            '    _state()["tool"] = True\n'
            '\n'
            '\n'
            'def last_cell():\n'
            '    """바로 앞에 실행한 \'내 셀\'의 기록 {"source", "stdout", "shown", "error", "blank"} 또는 None."""\n'
            '    rec = _state()["last"]\n'
            '    return dict(rec) if rec else None\n'
            '\n'
            '\n'
            'def forget():\n'
            '    """기록을 지운다(시험용)."""\n'
            '    state = _state()\n'
            '    state["last"] = None\n'
            '    state["tool"] = False\n'
            '\n'
            '\n'
            'def watching():\n'
            '    shell = _shell()\n'
            '    return shell is not None and _find_hooks(shell) is not None\n'
            '\n'
            '\n'
            'def _plain_text(shell, value):\n'
            '    if value is None:\n'
            '        return None\n'
            '    try:\n'
            '        data, _meta = shell.display_formatter.format(value, include=["text/plain"])\n'
            '        text = data.get("text/plain")\n'
            '    except Exception:\n'
            '        text = None\n'
            '    return None if text is None else _cap(str(text))\n'
            '\n'
            '\n'
            'def _make_hooks(shell):\n'
            '    state = _new_state()\n'
            '\n'
            '    def _z2h_before_cell(info=None, *_args, **_kwargs):\n'
            '        try:\n'
            '            tee = _Tee(sys.stdout)\n'
            '            state["stack"].append((sys.stdout, tee, getattr(info, "raw_cell", "") or ""))\n'
            '            state["tool"] = False\n'
            '            sys.stdout = tee\n'
            '        except Exception:\n'
            '            pass\n'
            '\n'
            '    def _z2h_after_cell(result=None, *_args, **_kwargs):\n'
            '        try:\n'
            '            if not state["stack"]:\n'
            '                return  # 우리 함수를 건 바로 그 셀: 셀 앞 알림을 받지 못했다\n'
            '            target, tee, source = state["stack"].pop()\n'
            '            if sys.stdout is tee:\n'
            '                sys.stdout = target\n'
            '            if state["tool"]:\n'
            '                state["tool"] = False\n'
            '                return\n'
            '            error = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)\n'
            '            _store(state, source, tee.text(), _plain_text(shell, getattr(result, "result", None)), error)\n'
            '        except Exception:\n'
            '            pass\n'
            '\n'
            '    for hook in (_z2h_before_cell, _z2h_after_cell):\n'
            '        hook.z2h_runs_state = state\n'
            '    return _z2h_before_cell, _z2h_after_cell\n'
            '\n'
            '\n'
            'def watch():\n'
            '    """셀 실행 기록을 켠다. 켰으면(또는 이미 켜져 있으면) True, IPython 이 아니라 못 켜면 False.\n'
            '    부른 셀은 도구 셀로 표시한다(준비 셀 자신을 \'내 셀\'로 기록하지 않게)."""\n'
            '    shell = _shell()\n'
            '    if shell is None:\n'
            '        return False\n'
            '    try:\n'
            '        if _find_hooks(shell) is None:\n'
            '            for event in EVENTS:  # 예전 모양의 우리 함수가 남아 있으면 뗀다\n'
            '                for cb in list(shell.events.callbacks.get(event, [])):\n'
            '                    if hasattr(cb, "z2h_runs_state"):\n'
            '                        shell.events.unregister(event, cb)\n'
            '            before, after = _make_hooks(shell)\n'
            '            shell.events.register("pre_run_cell", before)\n'
            '            shell.events.register("post_run_cell", after)\n'
            '        mark_tool_cell()\n'
            '        return True\n'
            '    except Exception:\n'
            '        return False\n'
        ),
    }
    packages = {'z2h_check', 'z2h_check.graders'}
    data = {
        'reference/l01.json': (
            '{\n'
            '  "lecture_id": "L01",\n'
            '  "version": 2,\n'
            '  "_설명": "1편 채점 기준값. 자동 생성 파일이라 손으로 고치지 않는다: python -m z2h_check.refgen --write (정본 inputs/pedagogy/content/L01.json). steps 는 정본의 코드 단계와 expected_check, rebuild 는 재구성 확인 셀과 통과 기준.",\n'
            '  "_출처": "정본 _meta.sources_and_licenses. 확인 식과 재구성 확인 셀의 강의 예제 숫자와 코드 동작 기준: karpathy/micrograd (MIT): Value 의 연산과 _backward, backward(위상 정렬), Neuron, Layer, MLP 구조(engine.py, nn.py) / karpathy/nn-zero-to-hero (MIT): lectures/micrograd 강의 노트북의 f(x) 예제, trace, draw_dot, lol, 뉴런 예제 숫자, 학습 반복문 / 우리가 새로 씀 (-): draw_text(그림 대신 글자로 보기), random.seed(1337) 줄, 확인 식(expected_check.expr), 재구성 확인 셀. LICENSE 전문은 third_party/licenses/, 표는 NOTICE.md.",\n'
            '  "canon": {\n'
            '    "file": "inputs/pedagogy/content/L01.json",\n'
            '    "version": "claude-1.0"\n'
            '  },\n'
            '  "tolerance": {\n'
            '    "number": 1e-06,\n'
            '    "_설명": "정본 _meta.code_conventions: output_match 는 contains, 소수는 이 허용오차까지 같으면 맞음"\n'
            '  },\n'
            '  "steps": [\n'
            '    {\n'
            '      "step_id": "L01-S02",\n'
            '      "title_ko": "도구 불러오기와 식 f",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 19.9999,\n'
            '        "max": 20.0001,\n'
            '        "expr": "f(3.0)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "f"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S03",\n'
            '      "title_ko": "f를 그래프로",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "20.0"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S04",\n'
            '      "title_ko": "조금 움직여 기울기 재기",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "14.003"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S05",\n'
            '      "title_ko": "입력이 셋인 식",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "4.0"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S06",\n'
            '      "title_ko": "a만 살짝 올리기",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "-3.0"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S07",\n'
            '      "title_ko": "Value 상자 설계도",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 1.9999,\n'
            '        "max": 2.0001,\n'
            '        "expr": "Value(2.0).data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S08",\n'
            '      "title_ko": "상자 세 개",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=2.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S09",\n'
            '      "title_ko": "더하기 가르치기",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=-1.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S10",\n'
            '      "title_ko": "곱하기 가르치기",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=4.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S11",\n'
            '      "title_ko": "어디서 왔는지 기억하기",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 2,\n'
            '        "max": 2,\n'
            '        "expr": "len(d._prev)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "d"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S12",\n'
            '      "title_ko": "이름표와 grad 칸",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=2.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S13",\n'
            '      "title_ko": "식 L까지 늘리기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=-8.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S14",\n'
            '      "title_ko": "그림으로 보기",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S15",\n'
            '      "title_ko": "맨 끝에서 시작",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S16",\n'
            '      "title_ko": "곱셈 규칙",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S17",\n'
            '      "title_ko": "숫자로 확인하는 lol 함수",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "-2.0"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S18",\n'
            '      "title_ko": "더하기는 그대로",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S19",\n'
            '      "title_ko": "연쇄법칙 한 번 더",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S20",\n'
            '      "title_ko": "기울기 쪽으로 한 걸음 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "-7.286496"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S21",\n'
            '      "title_ko": "tanh 모양",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S22",\n'
            '      "title_ko": "tanh 가르치기",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.7071,\n'
            '        "max": 0.7072,\n'
            '        "expr": "Value(0.8813735870195432).tanh().data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S23",\n'
            '      "title_ko": "뉴런의 입력과 가중치",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.8813,\n'
            '        "max": 0.8814,\n'
            '        "expr": "x1.data*w1.data + x2.data*w2.data + b.data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "b",\n'
            '        "w1",\n'
            '        "w2",\n'
            '        "x1",\n'
            '        "x2"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S24",\n'
            '      "title_ko": "뉴런 하나 계산 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S25",\n'
            '      "title_ko": "tanh를 지나 거꾸로",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S26",\n'
            '      "title_ko": "입력과 가중치까지 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S27",\n'
            '      "title_ko": "더하기의 _backward",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 1.9999,\n'
            '        "max": 2.0001,\n'
            '        "expr": "(lambda u, v: (lambda w: (setattr(w, \'grad\', 1.0), w._backward(), u.grad + v.grad)[-1])(u + v))(Value(1.0), Value(2.0))"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S28",\n'
            '      "title_ko": "곱하기와 tanh의 _backward (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 6.9999,\n'
            '        "max": 7.0001,\n'
            '        "expr": "(lambda u, v: (lambda w: (setattr(w, \'grad\', 1.0), w._backward(), u.grad + v.grad)[-1])(u * v))(Value(3.0), Value(4.0))"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S29",\n'
            '      "title_ko": "_backward를 하나씩 부르기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S30",\n'
            '      "title_ko": "줄 세우기: 위상 정렬 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 10,\n'
            '        "max": 10,\n'
            '        "expr": "len(topo)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "topo"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S31",\n'
            '      "title_ko": "backward()를 상자에 넣기",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.9999,\n'
            '        "max": 1.0001,\n'
            '        "expr": "(lambda v: (v.backward(), v.grad)[1])(Value(5.0))"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S32",\n'
            '      "title_ko": "한 줄로 역전파",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S33",\n'
            '      "title_ko": "같은 상자를 두 번 쓰면?",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S34",\n'
            '      "title_ko": "`=`를 `+=`로",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S35",\n'
            '      "title_ko": "숫자와 바로 계산하기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "Value(data=3.0)\\nValue(data=4.0)"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S36",\n'
            '      "title_ko": "exp 가르치기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 2.7182,\n'
            '        "max": 2.7183,\n'
            '        "expr": "Value(1.0).exp().data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S37",\n'
            '      "title_ko": "거듭제곱과 나누기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 7.9999,\n'
            '        "max": 8.0001,\n'
            '        "expr": "(Value(2.0)**3).data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S38",\n'
            '      "title_ko": "빼기, 음수, 오른쪽 더하기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 2.9999,\n'
            '        "max": 3.0001,\n'
            '        "expr": "(Value(5.0) - 2).data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S39",\n'
            '      "title_ko": "tanh를 쪼개 다시 만들기",\n'
            '      "expected_check": {\n'
            '        "type": "manual"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S40",\n'
            '      "title_ko": "PyTorch로 같은 뉴런 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "0.7071068"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S41",\n'
            '      "title_ko": "PyTorch 기울기와 비교",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "x2 0.5\\nw2 0.0\\nx1 -1.5\\nw1 1.0"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S42",\n'
            '      "title_ko": "Neuron 클래스 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 2,\n'
            '        "max": 2,\n'
            '        "expr": "len(Neuron(2).w)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Neuron"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S43",\n'
            '      "title_ko": "뉴런 써 보기",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": -1.0,\n'
            '        "max": 1.0,\n'
            '        "expr": "n(x).data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "n",\n'
            '        "x"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S44",\n'
            '      "title_ko": "Layer 클래스 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 3,\n'
            '        "max": 3,\n'
            '        "expr": "len(Layer(2, 3)(x))"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Layer",\n'
            '        "x"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S45",\n'
            '      "title_ko": "MLP 클래스 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 3,\n'
            '        "max": 3,\n'
            '        "expr": "len(MLP(3, [4, 4, 1]).layers)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "MLP"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S46",\n'
            '      "title_ko": "작은 신경망 그림",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": -1.0,\n'
            '        "max": 1.0,\n'
            '        "expr": "n(x).data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "n",\n'
            '        "x"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S47",\n'
            '      "title_ko": "작은 데이터 4개 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 4,\n'
            '        "max": 4,\n'
            '        "expr": "len(ypred)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "ypred"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S48",\n'
            '      "title_ko": "손실: 얼마나 틀렸나",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.0,\n'
            '        "max": 16.0,\n'
            '        "expr": "loss.data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "loss"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S49",\n'
            '      "title_ko": "손실에서 거꾸로",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 1,\n'
            '        "max": 41,\n'
            '        "expr": "sum(1 for l in n.layers for ne in l.neurons for p in ne.w + [ne.b] if p.grad != 0)"\n'
            '      },\n'
            '      "reads": [\n'
            '        "n"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S50",\n'
            '      "title_ko": "파라미터 모으기 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "output_match",\n'
            '        "expected": "41"\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S51",\n'
            '      "title_ko": "한 걸음 학습 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 6.0,\n'
            '        "max": 6.01\n'
            '      }\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S52",\n'
            '      "title_ko": "반복문으로 학습 (긴 단계)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 3.5,\n'
            '        "max": 4.5,\n'
            '        "expr": "loss.data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "loss"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S53",\n'
            '      "title_ko": "기울기 비우기(zero_grad)",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.0,\n'
            '        "max": 0.1,\n'
            '        "expr": "loss.data"\n'
            '      },\n'
            '      "reads": [\n'
            '        "loss"\n'
            '      ]\n'
            '    },\n'
            '    {\n'
            '      "step_id": "L01-S54",\n'
            '      "title_ko": "예측 보기",\n'
            '      "expected_check": {\n'
            '        "type": "value_range",\n'
            '        "min": 0.0,\n'
            '        "max": 0.1,\n'
            '        "expr": "sum((yp.data - yt)**2 for yp, yt in zip(ypred, ys))"\n'
            '      },\n'
            '      "reads": [\n'
            '        "ypred",\n'
            '        "ys"\n'
            '      ]\n'
            '    }\n'
            '  ],\n'
            '  "rebuild": [\n'
            '    {\n'
            '      "rebuild_id": "L01-R1",\n'
            '      "title_ko": "+, *, tanh와 각 _backward(+= 누적)가 있는 Value",\n'
            '      "check_code": "x1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); b = Value(6.8813735870195432)\\na1 = x1*w1; a2 = x2*w2; s = a1 + a2; n = s + b; o = n.tanh()\\no.grad = 1.0\\nfor v in (o, n, s, a2, a1):\\n  v._backward()\\nprint([round(v.grad, 4) for v in (x1, w1, x2, w2, b)])\\nc = Value(3.0); d = c * c; d.grad = 1.0; d._backward()\\nprint(d._prev == {c}, c.grad)",\n'
            '      "pass": {\n'
            '        "rule": "output_match",\n'
            '        "expected": "[-1.5, 1.0, 0.5, 0.0, 0.5]\\nTrue 6.0"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ],\n'
            '      "needs": [],\n'
            '      "max_hint_level_for_pass": 2\n'
            '    },\n'
            '    {\n'
            '      "rebuild_id": "L01-R2",\n'
            '      "title_ko": "위상 정렬을 쓰는 backward()",\n'
            '      "check_code": "a = Value(3.0)\\nb = a + a\\nb.backward()\\nprint(a.grad)\\nx1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); bb = Value(6.8813735870195432)\\no = (x1*w1 + x2*w2 + bb).tanh()\\no.backward()\\nprint([round(v.grad, 4) for v in (x1, w1, x2, w2, bb)])",\n'
            '      "pass": {\n'
            '        "rule": "output_match",\n'
            '        "expected": "2.0\\n[-1.5, 1.0, 0.5, 0.0, 0.5]"\n'
            '      },\n'
            '      "reads": [\n'
            '        "Value"\n'
            '      ],\n'
            '      "needs": [],\n'
            '      "max_hint_level_for_pass": 2\n'
            '    },\n'
            '    {\n'
            '      "rebuild_id": "L01-R3",\n'
            '      "title_ko": "학습 반복문(앞으로, 비우기, 거꾸로, 고치기)",\n'
            '      "check_code": "print(round(loss.data, 4), loss.data < 3.0)",\n'
            '      "pass": {\n'
            '        "rule": "ends_with",\n'
            '        "text": "True"\n'
            '      },\n'
            '      "reads": [\n'
            '        "loss"\n'
            '      ],\n'
            '      "needs": [\n'
            '        "MLP",\n'
            '        "random",\n'
            '        "xs",\n'
            '        "ys"\n'
            '      ],\n'
            '      "max_hint_level_for_pass": 2\n'
            '    }\n'
            '  ],\n'
            '  "mastery_rule": {\n'
            '    "mastery_rule_id": "default",\n'
            '    "steps_complete_ratio": 1.0,\n'
            '    "question_score_ratio": 0.8,\n'
            '    "rebuild_max_hint_level": 2,\n'
            '    "extra_checks": []\n'
            '  }\n'
            '}\n'
        ),
    }

    class _Z2HBundleFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        z2h_bundle = True

        def find_spec(self, fullname, path=None, target=None):
            if fullname in sources:
                return importlib.util.spec_from_loader(fullname, self, origin="<z2h_check 묶음>",
                                                       is_package=fullname in packages)
            return None

        def create_module(self, spec):
            return None

        def exec_module(self, module):
            name = module.__name__
            tail = "/__init__.py" if name in packages else ".py"
            filename = "<z2h_check 묶음>/" + name.replace(".", "/") + tail
            module.__file__ = filename
            if name == "z2h_check":
                module.__z2h_bundled__ = True
            exec(compile(sources[name], filename, "exec"), module.__dict__)
            if name == "z2h_check._resources":
                module._EMBEDDED.update(data)

    for finder in list(sys.meta_path):
        if getattr(finder, "z2h_bundle", False):
            sys.meta_path.remove(finder)
    for name in list(sys.modules):
        if name == "z2h_check" or name.startswith("z2h_check."):
            del sys.modules[name]
    sys.meta_path.insert(0, _Z2HBundleFinder())
    import z2h_check
    return z2h_check


if __name__ != "z2h_check._bundle":
    _z2h = _z2h_install()
    check, finish = _z2h.check, _z2h.finish
    print("z2h_check 준비 완료 (묶음 0.2.0): check(...)로 채점하고, 마지막에 finish(\"L01\")를 실행하세요.")
    del _z2h


## L01-S01 micrograd가 하는 일

영상: 챕터 1 「intro」 [0:00:00부터 0:00:25까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=0&end=25&cc_load_policy=1&cc_lang_pref=ko&hl=ko) · 챕터 2 「micrograd 개요」 [0:00:25부터 0:08:08까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=25&end=488&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 처음부터 8분 8초까지 봅니다. 이 단계에서는 코드를 치지 않습니다. 보면서 세 낱말만 기억하세요. 신경망(숫자를 받아 계산해서 답을 내는 큰 계산식), 기울기(입력을 조금 바꿀 때 출력이 얼마나 바뀌는지), 역전파(맨 끝 결과에서 앞쪽으로 거꾸로 가며 기울기를 구하는 방법). 영상 속 완성된 코드는 지금 따라 치지 않아도 됩니다. 다음 단계부터 같은 것을 처음부터 한 줄씩 만듭니다.

**보여야 하는 것**: 화면에 칠 것은 없습니다. 영상에서 완성된 micrograd(작은 자동 기울기 계산 도구)가 계산을 하고 기울기를 구하는 모습을 보면 됩니다.

이 단계는 노트북에서 칠 코드가 없습니다. 앱 화면에서 진행합니다.

## L01-S02 도구 불러오기와 식 f

영상: 챕터 3 「입력 1개 함수의 미분」 [0:08:08부터 0:14:12까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=488&end=852&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 8분 8초부터 보다가, 함수 f를 만드는 장면에서 멈춥니다. (1) 노트북 맨 위에 미리 들어 있는 '환경 확인' 준비 코드 셀을 실행합니다(없으면 아래 준비 코드를 새 셀에 붙여 넣고 실행). Colab과 맥미니 Jupyter 어느 쪽에서든 같은 노트북을 씁니다. (2) 다음 새 셀에 아래 5줄을 칩니다. 4번째 줄 끝의 쌍점(:)을 꼭 치세요. 5번째 줄은 앞에 빈칸(들여쓰기)을 둡니다. 화면은 2칸이지만, Enter를 치면 Colab은 2칸, 맥미니 Jupyter는 4칸을 알아서 넣어 줄 수 있습니다. 어느 쪽이든 됩니다.

**보여야 하는 것**: 준비 코드 셀은 세 줄을 찍습니다: 실행 장소(Colab 또는 내 컴퓨터), 기기(cpu, mps, cuda 중 하나), 그림 프로그램 graphviz(dot) 있음 또는 없음. 그림 프로그램이 '없음'이면 S14의 힌트 H2를 미리 읽어 두세요. 5줄짜리 셀은 아무것도 출력되지 않으면 정상입니다. 셀 왼쪽에 실행 번호(예: [2])가 생기면 끝난 것입니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
def f(x):
  return 3*x**2 - 4*x + 5

In [ ]:
check("L01-S02", globals())

## L01-S03 f를 그래프로

영상: 챕터 3 「입력 1개 함수의 미분」 [0:08:08부터 0:14:12까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=488&end=852&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 그래프를 그리는 장면을 본 뒤 새 셀에 4줄을 칩니다.

**보여야 하는 것**: 맨 위에 `20.0`이 찍히고, 아래에 U자(그릇) 모양 곡선이 보입니다. 그림 바로 위에 `[<matplotlib.lines.Line2D at 0x...>]` 같은 한 줄이 보여도 정상입니다(그림 정보를 글로 보여 준 것이고, 0x 뒤 숫자는 사람마다 다릅니다).

In [ ]:
print(f(3.0))
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
check("L01-S03", globals())

## L01-S04 조금 움직여 기울기 재기

영상: 챕터 3 「입력 1개 함수의 미분」 [0:08:08부터 0:14:12까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=488&end=852&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 h를 아주 작게 잡아 기울기를 재는 장면을 본 뒤 새 셀에 3줄을 칩니다. 마지막 줄은 print 없이 식만 씁니다. 셀의 마지막 줄이 식이면 노트북이 그 값을 보여 줍니다.

**보여야 하는 것**: `14.00300000000243`처럼 14에 아주 가까운 숫자가 보입니다. 끝자리는 조금 달라도 정상입니다.

In [ ]:
h = 0.001
x = 3.0
(f(x + h) - f(x)) / h

In [ ]:
check("L01-S04", globals())

## L01-S05 입력이 셋인 식

영상: 챕터 4 「입력 여러 개 함수의 미분」 [0:14:12부터 0:19:09까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=852&end=1149&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 14분 12초부터 봅니다. 입력이 a, b, c 세 개인 식을 만드는 장면 뒤에 새 셀에 5줄을 칩니다.

**보여야 하는 것**: `4.0`

In [ ]:
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

In [ ]:
check("L01-S05", globals())

## L01-S06 a만 살짝 올리기

영상: 챕터 4 「입력 여러 개 함수의 미분」 [0:14:12부터 0:19:09까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=852&end=1149&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 a만 조금 올려 d의 변화를 재는 장면을 본 뒤 새 셀에 5줄을 칩니다. 이 셀은 S05 셀 바로 다음에 실행합니다.

**보여야 하는 것**: `-3.000000000010772`처럼 -3에 아주 가까운 숫자. 끝자리는 조금 달라도 정상입니다.

In [ ]:
h = 0.0001
d1 = a*b + c
a += h
d2 = a*b + c
print((d2 - d1) / h)

In [ ]:
check("L01-S06", globals())

## L01-S07 Value 상자 설계도

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 19분 9초부터 봅니다. class Value를 처음 만드는 장면 뒤에 새 셀에 5줄을 칩니다. 이 셀을 앞으로 'Value 셀'이라고 부릅니다. 뒤 단계에서 여러 번 고칩니다. 줄 앞 빈칸(들여쓰기)은 화면에서 한 깊이에 2칸씩입니다. 맥미니 Jupyter처럼 Enter가 4칸씩 넣어 주면 4칸씩 써도 됩니다. 다만 뒤 단계에서 이 셀에 줄을 넣을 때는, 넣는 줄의 빈칸을 바로 위아래 같은 깊이 줄과 똑같이 맞춰야 합니다.

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다. 설계도만 만들었고 상자는 아직 없습니다.

In [ ]:
class Value:
  def __init__(self, data):
    self.data = data
  def __repr__(self):
    return f"Value(data={self.data})"

In [ ]:
check("L01-S07", globals())

## L01-S08 상자 세 개

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 Value(2.0)을 만드는 장면 뒤에 새 셀에 4줄을 칩니다. 이 셀을 '예시 셀'이라고 부릅니다.

**보여야 하는 것**: `Value(data=2.0)`

In [ ]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a

In [ ]:
check("L01-S08", globals())

## L01-S09 더하기 가르치기

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 __add__를 만드는 장면을 봅니다. 두 곳에 칩니다. (1) Value 셀(S07) 맨 아래에 3줄을 넣습니다. def 줄은 빈칸 2칸, 나머지는 4칸입니다. (2) Value 셀과 예시 셀(S08)을 차례로 다시 실행한 뒤, 새 셀에 마지막 1줄을 칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(__repr__의 return 줄 다음) · 3줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S08(예시 셀)

**보여야 하는 것**: `Value(data=-1.0)`

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S09: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data):
    self.data = data
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data)
    return out

In [ ]:
# L01-S09: 다시 실행할 L01-S08(예시 셀)과(와) 같은 코드
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a

In [ ]:
a + b

In [ ]:
check("L01-S09", globals())

## L01-S10 곱하기 가르치기

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 __mul__을 만드는 장면을 봅니다. Value 셀 맨 아래에 3줄을 넣고, Value 셀과 예시 셀을 다시 실행한 뒤, 새 셀에 2줄을 칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(__add__의 return out 다음) · 3줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S08(예시 셀)

**보여야 하는 것**: `Value(data=4.0)`. S05의 `4.0`과 같은 답입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S10: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data):
    self.data = data
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data)
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data)
    return out

In [ ]:
# L01-S10: 다시 실행할 L01-S08(예시 셀)과(와) 같은 코드
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a

In [ ]:
d = a*b + c
d

In [ ]:
check("L01-S10", globals())

## L01-S11 어디서 왔는지 기억하기

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 _children과 _op를 넣는 장면을 봅니다. Value 셀의 네 곳을 고칩니다. (1) __init__ 줄을 바꾸고 (2) self.data = data 아래에 2줄을 넣고 (3) __add__의 out = 줄과 (4) __mul__의 out = 줄을 바꿉니다. 그다음 Value 셀, 예시 셀, S10 셀을 차례로 다시 실행하고 새 셀에 1줄을 칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): __init__ 줄을 통째로 바꿉니다 · 1줄 바꾸기
- L01-S07(Value 셀): self.data = data 줄 바로 아래 · 2줄 넣기
- L01-S07(Value 셀): __add__ 안의 out = 줄을 바꿉니다 · 1줄 바꾸기
- L01-S07(Value 셀): __mul__ 안의 out = 줄을 바꿉니다 · 1줄 바꾸기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S08(예시 셀) → L01-S10

**보여야 하는 것**: `({Value(data=-6.0), Value(data=10.0)}, '+')`. 중괄호 안 두 상자의 순서는 반대로 나와도 정상입니다(묶음 set은 순서를 정하지 않습니다).

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S11: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op=''):
    self.data = data
    self._prev = set(_children)
    self._op = _op
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    return out

In [ ]:
# L01-S11: 다시 실행할 L01-S08(예시 셀)과(와) 같은 코드
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
a

In [ ]:
# L01-S11: 다시 실행할 L01-S10과(와) 같은 코드
d = a*b + c
d

In [ ]:
d._prev, d._op

In [ ]:
check("L01-S11", globals())

## L01-S12 이름표와 grad 칸

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 label과 grad를 넣는 장면을 봅니다. Value 셀의 세 곳과 예시 셀의 세 줄을 고친 뒤, Value 셀과 예시 셀을 다시 실행합니다. 이 단계에는 새 셀이 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): __init__ 줄을 통째로 바꿉니다 · 1줄 바꾸기
- L01-S07(Value 셀): self.data = data 줄 바로 아래 · 1줄 넣기
- L01-S07(Value 셀): self._op = _op 줄 바로 아래 · 1줄 넣기
- L01-S08(예시 셀): 예시 셀의 a = 줄 · 1줄 바꾸기
- L01-S08(예시 셀): 예시 셀의 b = 줄 · 1줄 바꾸기
- L01-S08(예시 셀): 예시 셀의 c = 줄 · 1줄 바꾸기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S08(예시 셀)

**보여야 하는 것**: 예시 셀을 다시 실행하면 에러 없이 `Value(data=2.0)`이 보입니다. 겉모습은 그대로지만 상자 안에 이름표와 grad 칸이 생겼습니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S12: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    return out

In [ ]:
# L01-S12: L01-S08(예시 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
a

In [ ]:
check("L01-S12", globals())

## L01-S13 식 L까지 늘리기 (긴 단계)

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 e, d, f, L을 만드는 장면을 본 뒤 새 셀에 8줄을 칩니다. 주의: 여기서 f라는 이름을 상자에 다시 쓰므로 S02의 함수 f는 사라집니다. 앞으로 함수 f는 쓰지 않으니 괜찮습니다.

**보여야 하는 것**: `Value(data=-8.0)`. 계산: e = 2 x (-3) = -6, d = -6 + 10 = 4, L = 4 x (-2) = -8.

In [ ]:
e = a*b
e.label = 'e'
d = e + c
d.label = 'd'
f = Value(-2.0, label='f')
L = d * f
L.label = 'L'
L

In [ ]:
check("L01-S13", globals())

## L01-S14 그림으로 보기

영상: 챕터 5 「Value 객체 시작, 시각화(draw_dot)」 [0:19:09부터 0:32:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1149&end=1930&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 draw_dot으로 그림을 그리는 장면을 봅니다. 그림 그리는 코드는 길어서 직접 치지 않습니다. (1) 노트북에 미리 들어 있는 그림 준비 코드 셀을 실행합니다(없으면 아래 준비 코드를 새 셀에 붙여 넣고 실행). (2) 다음 새 셀에 1줄을 칩니다.

**보여야 하는 것**: 왼쪽에서 오른쪽으로 흐르는 그림이 나옵니다. 스스로 확인할 것: (1) 네모가 7개(a, b, c, e, d, f, L)이다. (2) 동그라미가 3개(*, +, *)이고 연산을 나타낸다. (3) 맨 오른쪽 L 칸이 `L | data -8.0000 | grad 0.0000`이다. (4) 모든 칸의 grad가 0.0000이다. 그림 대신 에러가 나면 힌트 H2를 보세요.

In [ ]:
from graphviz import Digraph
def trace(root):
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges
def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      dot.node(name=uid + n._op, label=n._op)
      dot.edge(uid + n._op, uid)
  for n1, n2 in edges:
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)
  return dot
def draw_text(root):
  nodes, edges = trace(root)
  for n in nodes:
    print(n.label, '| data %.4f | grad %.4f |' % (n.data, n.grad), n._op, [c.label for c in n._prev])

In [ ]:
draw_dot(L)

In [ ]:
check("L01-S14", globals())

## L01-S15 맨 끝에서 시작

영상: 챕터 6 「손으로 역전파 예제 1: 간단한 식」 [0:32:10부터 0:51:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 32분 10초부터 봅니다. 맨 끝 L의 기울기를 정하는 장면 뒤에 새 셀에 2줄을 칩니다.

**보여야 하는 것**: 스스로 확인할 것: (1) L 칸이 `grad 1.0000`이다. (2) 나머지 여섯 칸은 모두 `grad 0.0000`이다.

In [ ]:
L.grad = 1.0
draw_dot(L)

In [ ]:
check("L01-S15", globals())

## L01-S16 곱셈 규칙

영상: 챕터 6 「손으로 역전파 예제 1: 간단한 식」 [0:32:10부터 0:51:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 L = d * f를 미분하는 장면을 봅니다. 영상에서 손으로 구한 두 값을 새 셀에 적습니다(3줄).

**보여야 하는 것**: 스스로 확인할 것: (1) f 칸 `grad 4.0000` (2) d 칸 `grad -2.0000` (3) L 칸 `grad 1.0000` (4) a, b, c, e는 아직 `grad 0.0000`.

In [ ]:
f.grad = 4.0
d.grad = -2.0
draw_dot(L)

In [ ]:
check("L01-S16", globals())

## L01-S17 숫자로 확인하는 lol 함수

영상: 챕터 6 「손으로 역전파 예제 1: 간단한 식」 [0:32:10부터 0:51:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 lol 함수로 기울기를 숫자로 확인하는 장면을 봅니다. (1) 노트북에 미리 들어 있는 lol 준비 코드 셀을 실행합니다(없으면 아래 준비 코드를 새 셀에 붙여 넣고 실행. 함수를 만들기만 해서 출력 없음). (2) 다음 새 셀에 1줄을 쳐서 함수를 부릅니다. 더 해 보고 싶으면: 준비 코드의 `  d.data += h` 줄을 지우고 `  f = Value(-2.0)` 줄 바로 아래에 `  f.data += h`를 넣어 다시 실행해 보세요. 약 4.0이 나옵니다.

**보여야 하는 것**: `-2.000000000000668`처럼 -2에 아주 가까운 숫자. 끝자리는 조금 달라도 정상입니다(컴퓨터 소수 계산의 작은 오차).

In [ ]:
def lol():
  h = 0.001
  a = Value(2.0)
  b = Value(-3.0)
  c = Value(10.0)
  e = a*b
  d = e + c
  f = Value(-2.0)
  L = d * f
  L1 = L.data
  a = Value(2.0)
  b = Value(-3.0)
  c = Value(10.0)
  e = a*b
  d = e + c
  d.data += h
  f = Value(-2.0)
  L = d * f
  L2 = L.data
  print((L2 - L1) / h)

In [ ]:
lol()

In [ ]:
check("L01-S17", globals())

## L01-S18 더하기는 그대로

영상: 챕터 6 「손으로 역전파 예제 1: 간단한 식」 [0:32:10부터 0:51:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 c와 e의 기울기를 구하는 장면(연쇄법칙)을 봅니다. 새 셀에 3줄을 칩니다.

**보여야 하는 것**: 스스로 확인할 것: (1) c 칸과 e 칸이 `grad -2.0000` (2) d -2.0000, f 4.0000, L 1.0000은 그대로 (3) a, b는 아직 `grad 0.0000`.

In [ ]:
c.grad = -2.0
e.grad = -2.0
draw_dot(L)

In [ ]:
check("L01-S18", globals())

## L01-S19 연쇄법칙 한 번 더

영상: 챕터 6 「손으로 역전파 예제 1: 간단한 식」 [0:32:10부터 0:51:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=1930&end=3070&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 a와 b의 기울기를 구하는 장면을 봅니다. 새 셀에 3줄을 칩니다. 곱하기를 식으로 그대로 써서, 어디서 온 숫자인지 보이게 합니다.

**보여야 하는 것**: 스스로 확인할 것: (1) a 칸 `grad 6.0000` (2) b 칸 `grad -4.0000` (3) 이제 일곱 칸 모두 grad가 0이 아니다.

In [ ]:
a.grad = -2.0 * -3.0
b.grad = -2.0 * 2.0
draw_dot(L)

In [ ]:
check("L01-S19", globals())

## L01-S20 기울기 쪽으로 한 걸음 (긴 단계)

영상: 챕터 7 「최적화 한 단계 미리보기」 [0:51:10부터 0:52:52까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3070&end=3172&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 51분 10초부터 봅니다. 입력을 기울기 방향으로 조금씩 움직이는 장면 뒤에 새 셀에 8줄을 칩니다. 이 셀은 한 번만 실행하세요. 다시 실행하면 한 걸음 더 움직여 숫자가 달라집니다.

**보여야 하는 것**: `-7.286496`처럼 -7.29에 가까운 숫자. 처음의 -8.0보다 커졌습니다.

In [ ]:
a.data += 0.01 * a.grad
b.data += 0.01 * b.grad
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad
e = a * b
d = e + c
L = d * f
print(L.data)

In [ ]:
check("L01-S20", globals())

## L01-S21 tanh 모양

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 52분 52초부터 봅니다. 뉴런(신경망의 한 칸) 이야기와 tanh 그래프 장면을 본 뒤 새 셀에 2줄을 칩니다.

**보여야 하는 것**: S자 곡선과 격자무늬가 나옵니다. 스스로 확인할 것: (1) 왼쪽 끝은 -1에, 오른쪽 끝은 1에 거의 붙어 있다. (2) 가운데(x = 0)에서 0을 지난다. x가 2만 돼도 tanh는 약 0.96으로 거의 1입니다.

In [ ]:
plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2)))
plt.grid()

In [ ]:
check("L01-S21", globals())

## L01-S22 tanh 가르치기

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 Value에 tanh를 넣는 장면을 봅니다. Value 셀 맨 아래에 5줄을 넣고 Value 셀을 다시 실행합니다. 새 셀은 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(__mul__의 return out 다음) · 5줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S22: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    return out

In [ ]:
check("L01-S22", globals())

## L01-S23 뉴런의 입력과 가중치

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 x1, x2, w1, w2, b를 만드는 장면 뒤에 새 셀에 5줄을 칩니다. 이 셀을 '뉴런 입력 셀'이라고 부릅니다. b의 숫자는 길지만 영상과 같게 그대로 칩니다.

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

In [ ]:
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

In [ ]:
check("L01-S23", globals())

## L01-S24 뉴런 하나 계산 (긴 단계)

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 뉴런을 계산해 그림을 그리는 장면 뒤에 새 셀에 6줄을 칩니다. 이 셀을 '뉴런 계산 셀'이라고 부릅니다. 세미콜론(;)은 한 줄에 두 명령을 이어 쓰는 표시입니다. 계산 뒤 바로 이름표를 붙이려고 씁니다.

**보여야 하는 것**: 스스로 확인할 것: (1) 맨 오른쪽 o 칸이 `o | data 0.7071 | grad 0.0000` (2) n 칸이 `data 0.8814` (3) x1*w1 칸 `data -6.0000`, x2*w2 칸 `data 0.0000` (4) grad는 아직 모두 0.

In [ ]:
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
draw_dot(o)

In [ ]:
check("L01-S24", globals())

## L01-S25 tanh를 지나 거꾸로

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 o에서 n으로 기울기를 넘기는 장면을 봅니다. 새 셀에 3줄을 칩니다.

**보여야 하는 것**: 스스로 확인할 것: (1) o 칸 `grad 1.0000` (2) n 칸 `grad 0.5000` (3) 나머지는 아직 `grad 0.0000`.

In [ ]:
o.grad = 1.0
n.grad = 1 - o.data**2
draw_dot(o)

In [ ]:
check("L01-S25", globals())

## L01-S26 입력과 가중치까지 (긴 단계)

영상: 챕터 8 「손으로 역전파 예제 2: 뉴런」 [0:52:52부터 1:09:02까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=3172&end=4142&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 더하기 칸 두 개와 곱하기 칸 두 개를 지나 기울기를 넘기는 장면을 봅니다. 새 셀에 9줄을 칩니다.

**보여야 하는 것**: 스스로 확인할 것: (1) x1 `grad -1.5000` (2) w1 `grad 1.0000` (3) x2 `grad 0.5000` (4) w2 `grad 0.0000` (5) b, n, x1*w1, x2*w2, x1*w1 + x2*w2 칸은 모두 `grad 0.5000`.

In [ ]:
x1w1x2w2.grad = 0.5
b.grad = 0.5
x1w1.grad = 0.5
x2w2.grad = 0.5
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad
draw_dot(o)

In [ ]:
check("L01-S26", globals())

## L01-S27 더하기의 _backward

영상: 챕터 9 「연산별 _backward 구현」 [1:09:02부터 1:17:32까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4142&end=4652&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 9분 2초부터 봅니다. 기울기 넘기는 일을 연산마다 맡기는 장면입니다. Value 셀의 두 곳에 넣고 Value 셀을 다시 실행합니다. (1) self.grad = 0.0 아래에 1줄 (2) __add__의 out = 줄 아래에 4줄. 여기서는 영상대로 `=`를 씁니다. 이 `=`에는 숨은 버그가 있고, S34에서 고칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): self.grad = 0.0 줄 바로 아래 · 1줄 넣기
- L01-S07(Value 셀): __add__ 안의 out = 줄 바로 아래(return out 위) · 4줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S27: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad = 1.0 * out.grad
      other.grad = 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    return out

In [ ]:
check("L01-S27", globals())

## L01-S28 곱하기와 tanh의 _backward (긴 단계)

영상: 챕터 9 「연산별 _backward 구현」 [1:09:02부터 1:17:32까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4142&end=4652&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 곱하기와 tanh의 _backward를 만드는 장면을 봅니다. Value 셀의 두 곳에 넣고 Value 셀을 다시 실행합니다. (1) __mul__의 out = 줄 아래에 4줄 (2) tanh의 out = 줄 아래에 3줄.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): __mul__ 안의 out = 줄 바로 아래(return out 위) · 4줄 넣기
- L01-S07(Value 셀): tanh 안의 out = 줄 바로 아래(return out 위) · 3줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S28: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad = 1.0 * out.grad
      other.grad = 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad = other.data * out.grad
      other.grad = self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad = (1 - t**2) * out.grad
    out._backward = _backward
    return out

In [ ]:
check("L01-S28", globals())

## L01-S29 _backward를 하나씩 부르기 (긴 단계)

영상: 챕터 9 「연산별 _backward 구현」 [1:09:02부터 1:17:32까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4142&end=4652&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 _backward를 뒤에서부터 차례로 부르는 장면을 봅니다. 먼저 Value 셀, 뉴런 입력 셀(S23), 뉴런 계산 셀(S24)을 차례로 다시 실행해 grad를 0으로 되돌립니다. 그다음 새 셀에 7줄을 칩니다.

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S23(뉴런 입력 셀) → L01-S24(뉴런 계산 셀)

**보여야 하는 것**: 스스로 확인할 것: S26과 같은 숫자인가? x1 `grad -1.5000`, w1 `grad 1.0000`, x2 `grad 0.5000`, w2 `grad 0.0000`, n과 b `grad 0.5000`.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S29: 다시 실행할 L01-S07(Value 셀)과(와) 같은 코드
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad = 1.0 * out.grad
      other.grad = 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad = other.data * out.grad
      other.grad = self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad = (1 - t**2) * out.grad
    out._backward = _backward
    return out

In [ ]:
# L01-S29: 다시 실행할 L01-S23(뉴런 입력 셀)과(와) 같은 코드
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

In [ ]:
# L01-S29: 다시 실행할 L01-S24(뉴런 계산 셀)과(와) 같은 코드
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
draw_dot(o)

In [ ]:
o.grad = 1.0
o._backward()
n._backward()
x1w1x2w2._backward()
x2w2._backward()
x1w1._backward()
draw_dot(o)

In [ ]:
check("L01-S29", globals())

## L01-S30 줄 세우기: 위상 정렬 (긴 단계)

영상: 챕터 10 「그래프 전체 backward() 구현」 [1:17:32부터 1:22:28까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4652&end=4948&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 17분 32초부터 봅니다. topo 목록을 만드는 장면 뒤에 새 셀에 10줄을 칩니다. 위상 정렬은 '어떤 상자보다 그 재료가 늘 앞에 오도록' 줄을 세우는 방법입니다.

**보여야 하는 것**: `[Value(data=...), ..., Value(data=0.7071067811865476)]` 모양으로 상자 10개가 든 목록. 맨 끝은 늘 o(0.7071...)입니다. 앞쪽 순서는 사람마다 조금 다를 수 있습니다.

In [ ]:
topo = []
visited = set()
def build_topo(v):
  if v not in visited:
    visited.add(v)
    for child in v._prev:
      build_topo(child)
    topo.append(v)
build_topo(o)
topo

In [ ]:
check("L01-S30", globals())

## L01-S31 backward()를 상자에 넣기

영상: 챕터 10 「그래프 전체 backward() 구현」 [1:17:32부터 1:22:28까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4652&end=4948&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 backward()를 Value 안에 넣는 장면을 봅니다. Value 셀 맨 아래에 13줄을 넣는데, 그중 8줄은 S30 셀에서 복사해 옵니다. (1) `  def backward(self):`를 칩니다. (2) S30의 첫 8줄(topo = []부터 topo.append(v)까지)을 복사해 붙이고, 줄마다 두 깊이만큼 더 들여씁니다(화면 기준 빈칸 4칸, 4칸씩 쓰는 셀이면 8칸). 여러 줄을 고른 뒤 Tab 키를 누르면 한꺼번에 들여쓸 수 있습니다. 화면처럼 맞을 때까지 누르세요. (3) 나머지 4줄을 칩니다. 그다음 Value 셀을 다시 실행합니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(tanh의 return out 다음) · 13줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S31: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad = 1.0 * out.grad
      other.grad = 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad = other.data * out.grad
      other.grad = self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad = (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
check("L01-S31", globals())

## L01-S32 한 줄로 역전파

영상: 챕터 10 「그래프 전체 backward() 구현」 [1:17:32부터 1:22:28까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4652&end=4948&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 o.backward()를 부르는 장면을 봅니다. Value 셀, 뉴런 입력 셀(S23), 뉴런 계산 셀(S24)을 차례로 다시 실행한 뒤 새 셀에 2줄을 칩니다.

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S23(뉴런 입력 셀) → L01-S24(뉴런 계산 셀)

**보여야 하는 것**: 스스로 확인할 것: S26, S29와 같은 숫자인가? x1 `grad -1.5000`, w1 `grad 1.0000`, x2 `grad 0.5000`, w2 `grad 0.0000`, n과 b `grad 0.5000`, o `grad 1.0000`.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S32: 다시 실행할 L01-S07(Value 셀)과(와) 같은 코드
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad = 1.0 * out.grad
      other.grad = 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad = other.data * out.grad
      other.grad = self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad = (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
# L01-S32: 다시 실행할 L01-S23(뉴런 입력 셀)과(와) 같은 코드
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

In [ ]:
# L01-S32: 다시 실행할 L01-S24(뉴런 계산 셀)과(와) 같은 코드
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
draw_dot(o)

In [ ]:
o.backward()
draw_dot(o)

In [ ]:
check("L01-S32", globals())

## L01-S33 같은 상자를 두 번 쓰면?

영상: 챕터 11 「노드를 여러 번 쓸 때의 버그 수정」 [1:22:28부터 1:27:05까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4948&end=5225&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 22분 28초부터 봅니다. b = a + a에서 기울기가 틀리는 장면입니다. 새 셀에 5줄을 칩니다. 이번 결과는 일부러 틀린 값이 나옵니다. 왜 틀렸는지 보는 단계입니다.

**보여야 하는 것**: 스스로 확인할 것: (1) a 칸이 `a | data 3.0000 | grad 1.0000` (2) b 칸이 `b | data 6.0000 | grad 1.0000`. b = 2a이므로 a의 정답은 2.0입니다. 틀린 값 1.0이 나오는 것이 이 단계의 정상입니다.

In [ ]:
a = Value(3.0, label='a')
b = a + a
b.label = 'b'
b.backward()
draw_dot(b)

In [ ]:
check("L01-S33", globals())

## L01-S34 `=`를 `+=`로

영상: 챕터 11 「노드를 여러 번 쓸 때의 버그 수정」 [1:22:28부터 1:27:05까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=4948&end=5225&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 버그를 고치는 장면을 봅니다. Value 셀의 _backward 안 다섯 줄에서 `=`를 `+=`로 바꿉니다(더하기 2줄, 곱하기 2줄, tanh 1줄). 그다음 Value 셀과 S33 셀을 차례로 다시 실행합니다. 새 셀은 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): __add__의 _backward 첫 줄 · 1줄 바꾸기
- L01-S07(Value 셀): __add__의 _backward 둘째 줄 · 1줄 바꾸기
- L01-S07(Value 셀): __mul__의 _backward 첫 줄 · 1줄 바꾸기
- L01-S07(Value 셀): __mul__의 _backward 둘째 줄 · 1줄 바꾸기
- L01-S07(Value 셀): tanh의 _backward 줄 · 1줄 바꾸기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S33

**보여야 하는 것**: 스스로 확인할 것: S33 셀을 다시 실행한 그림에서 a 칸이 `grad 2.0000`이다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S34: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
# L01-S34: 다시 실행할 L01-S33과(와) 같은 코드
a = Value(3.0, label='a')
b = a + a
b.label = 'b'
b.backward()
draw_dot(b)

In [ ]:
check("L01-S34", globals())

## L01-S35 숫자와 바로 계산하기 (긴 단계)

영상: 챕터 12 「tanh 쪼개기, 연산 더 추가」 [1:27:05부터 1:39:31까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 27분 5초부터 봅니다. Value(2.0) + 1처럼 그냥 숫자와도 계산되게 하는 장면입니다. Value 셀의 세 곳에 넣습니다. (1) __add__ 첫 줄 아래 1줄 (2) __mul__ 첫 줄 아래 1줄 (3) 셀 맨 아래 2줄. Value 셀을 다시 실행한 뒤 새 셀에 3줄을 칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): __add__의 def 줄 바로 아래 · 1줄 넣기
- L01-S07(Value 셀): __mul__의 def 줄 바로 아래 · 1줄 넣기
- L01-S07(Value 셀): Value 셀 맨 아래(backward의 마지막 줄 다음) · 2줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 두 줄: `Value(data=3.0)`과 `Value(data=4.0)`

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S35: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
  def __rmul__(self, other):
    return self * other

In [ ]:
a = Value(2.0)
print(a + 1)
print(2 * a)

In [ ]:
check("L01-S35", globals())

## L01-S36 exp 가르치기 (긴 단계)

영상: 챕터 12 「tanh 쪼개기, 연산 더 추가」 [1:27:05부터 1:39:31까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 exp를 만드는 장면을 봅니다. Value 셀 맨 아래에 7줄을 넣고 Value 셀을 다시 실행합니다. 새 셀은 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(__rmul__ 다음) · 7줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S36: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
  def __rmul__(self, other):
    return self * other
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

In [ ]:
check("L01-S36", globals())

## L01-S37 거듭제곱과 나누기 (긴 단계)

영상: 챕터 12 「tanh 쪼개기, 연산 더 추가」 [1:27:05부터 1:39:31까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 __pow__와 __truediv__를 만드는 장면을 봅니다. Value 셀 맨 아래에 9줄을 넣고 Value 셀을 다시 실행합니다. 새 셀은 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(exp 다음) · 9줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S37: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
  def __rmul__(self, other):
    return self * other
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "지수는 그냥 숫자만 됩니다"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward
    return out
  def __truediv__(self, other):
    return self * other**-1

In [ ]:
check("L01-S37", globals())

## L01-S38 빼기, 음수, 오른쪽 더하기 (긴 단계)

영상: 챕터 12 「tanh 쪼개기, 연산 더 추가」 [1:27:05부터 1:39:31까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 빼기와 음수 만드는 장면을 봅니다. Value 셀 맨 아래에 6줄을 넣고 Value 셀을 다시 실행합니다. 새 셀은 없습니다. __radd__는 이 구간보다 뒤(1시간 54분 근처)에서 필요해지는 것을 미리 넣습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S07(Value 셀): Value 셀 맨 아래(__truediv__ 다음) · 6줄 넣기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀)

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S38: L01-S07(Value 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
  def __rmul__(self, other):
    return self * other
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "지수는 그냥 숫자만 됩니다"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward
    return out
  def __truediv__(self, other):
    return self * other**-1
  def __neg__(self):
    return self * -1
  def __sub__(self, other):
    return self + (-other)
  def __radd__(self, other):
    return self + other

In [ ]:
check("L01-S38", globals())

## L01-S39 tanh를 쪼개 다시 만들기

영상: 챕터 12 「tanh 쪼개기, 연산 더 추가」 [1:27:05부터 1:39:31까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5225&end=5971&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 tanh를 exp와 나누기로 바꿔 쓰는 장면을 봅니다. 뉴런 계산 셀(S24)의 두 곳을 고칩니다. (1) n = 줄 바로 아래에 e 줄을 넣고 (2) o = 줄을 바꿉니다. 그다음 Value 셀, 뉴런 입력 셀(S23), 뉴런 계산 셀(S24)을 차례로 다시 실행하고 새 셀에 2줄을 칩니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S24(뉴런 계산 셀): n = 줄 바로 아래 · 1줄 넣기
- L01-S24(뉴런 계산 셀): o = 줄을 바꿉니다 · 1줄 바꾸기

**다시 실행할 셀(차례로)**: L01-S07(Value 셀) → L01-S23(뉴런 입력 셀) → L01-S24(뉴런 계산 셀)

**보여야 하는 것**: 그림이 훨씬 커집니다(exp, +, **-1, * 같은 칸이 늘어남). 스스로 확인할 것: (1) x1 `grad -1.5000`, w1 `grad 1.0000`, x2 `grad 0.5000`, w2 `grad 0.0000`으로 S32와 같다. (2) o 칸의 data가 0.7071 그대로다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S39: 다시 실행할 L01-S07(Value 셀)과(와) 같은 코드
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()
  def __rmul__(self, other):
    return self * other
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "지수는 그냥 숫자만 됩니다"
    out = Value(self.data**other, (self,), f'**{other}')
    def _backward():
      self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward
    return out
  def __truediv__(self, other):
    return self * other**-1
  def __neg__(self):
    return self * -1
  def __sub__(self, other):
    return self + (-other)
  def __radd__(self, other):
    return self + other

In [ ]:
# L01-S39: 다시 실행할 L01-S23(뉴런 입력 셀)과(와) 같은 코드
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')

In [ ]:
# L01-S39: L01-S24(뉴런 계산 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
e = (2*n).exp()
o = (e - 1) / (e + 1); o.label = 'o'
draw_dot(o)

In [ ]:
o.backward()
draw_dot(o)

In [ ]:
check("L01-S39", globals())

## L01-S40 PyTorch로 같은 뉴런 (긴 단계)

영상: 챕터 13 「PyTorch로 같은 일 하기」 [1:39:31부터 1:43:55까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5971&end=6235&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 39분 31초부터 봅니다. PyTorch(실제로 많이 쓰는 신경망 도구)로 같은 뉴런을 만드는 장면 뒤에 새 셀에 9줄을 칩니다. 텐서(tensor)는 PyTorch의 숫자 상자입니다. 여기서는 숫자 하나짜리 텐서를 씁니다.

**보여야 하는 것**: `0.7071067811865476`. S24의 o와 같은 숫자입니다. 끝자리는 조금 달라도 정상입니다.

In [ ]:
import torch
x1 = torch.tensor([2.0], dtype=torch.double, requires_grad=True)
x2 = torch.tensor([0.0], dtype=torch.double, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.double, requires_grad=True)
w2 = torch.tensor([1.0], dtype=torch.double, requires_grad=True)
b = torch.tensor([6.8813735870195432], dtype=torch.double, requires_grad=True)
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)
print(o.item())

In [ ]:
check("L01-S40", globals())

## L01-S41 PyTorch 기울기와 비교

영상: 챕터 13 「PyTorch로 같은 일 하기」 [1:39:31부터 1:43:55까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=5971&end=6235&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 PyTorch의 기울기를 꺼내 보는 장면 뒤에 새 셀에 5줄을 칩니다.

**보여야 하는 것**: 네 줄: `x2 0.49999999999999994`, `w2 0.0`, `x1 -1.4999999999999998`, `w1 0.9999999999999999`. 곧 0.5, 0.0, -1.5, 1.0입니다. 끝자리의 작은 차이는 컴퓨터 소수 계산의 오차라 정상입니다.

In [ ]:
o.backward()
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

In [ ]:
check("L01-S41", globals())

## L01-S42 Neuron 클래스 (긴 단계)

영상: 챕터 14 「MLP 라이브러리 만들기」 [1:43:55부터 1:51:04까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 43분 55초부터 봅니다. class Neuron을 만드는 장면 뒤에 새 셀에 9줄을 칩니다. 이 셀을 'Neuron 셀'이라고 부릅니다.

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

In [ ]:
import random
class Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1, 1))
  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out

In [ ]:
check("L01-S42", globals())

## L01-S43 뉴런 써 보기

영상: 챕터 14 「MLP 라이브러리 만들기」 [1:43:55부터 1:51:04까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 뉴런을 불러 보는 장면 뒤에 새 셀에 3줄을 칩니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. `Value(data=0.53...)`처럼 -1과 1 사이 숫자 하나를 담은 상자가 보이면 정상입니다.

In [ ]:
x = [2.0, 3.0]
n = Neuron(2)
n(x)

In [ ]:
check("L01-S43", globals())

## L01-S44 Layer 클래스 (긴 단계)

영상: 챕터 14 「MLP 라이브러리 만들기」 [1:43:55부터 1:51:04까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 class Layer를 만드는 장면 뒤에 새 셀에 7줄을 칩니다. 이 셀을 'Layer 셀'이라고 부릅니다. 마지막 줄은 만든 층을 바로 써 보는 줄입니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. `[Value(data=...), Value(data=...), Value(data=...)]`처럼 상자 3개가 든 목록이 보이면 정상입니다.

In [ ]:
class Layer:
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
Layer(2, 3)(x)

In [ ]:
check("L01-S44", globals())

## L01-S45 MLP 클래스 (긴 단계)

영상: 챕터 14 「MLP 라이브러리 만들기」 [1:43:55부터 1:51:04까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 class MLP(다층 신경망)를 만드는 장면 뒤에 새 셀에 8줄을 칩니다. 이 셀을 'MLP 셀'이라고 부릅니다.

**보여야 하는 것**: 아무것도 출력되지 않으면 정상입니다.

In [ ]:
class MLP:
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x

In [ ]:
check("L01-S45", globals())

## L01-S46 작은 신경망 그림

영상: 챕터 14 「MLP 라이브러리 만들기」 [1:43:55부터 1:51:04까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6235&end=6664&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 MLP(3, [4, 4, 1])을 만들고 그림을 그리는 장면 뒤에 새 셀에 4줄을 칩니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. 첫 줄에 -1과 1 사이 숫자 하나를 담은 `Value(data=...)`, 그 아래에 아주 큰 그림이 나옵니다. 그림이 커서 옆으로 길게 밀어 봐야 할 수 있습니다.

In [ ]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
print(n(x))
draw_dot(n(x))

In [ ]:
check("L01-S46", globals())

## L01-S47 작은 데이터 4개 (긴 단계)

영상: 챕터 15 「작은 데이터셋, 손실 함수」 [1:51:04부터 1:57:56까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6664&end=7076&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 51분 4초부터 봅니다. 입력 4개와 바라는 답 4개를 만드는 장면 뒤에 새 셀에 7줄을 칩니다. 첫 줄부터 넷째 줄까지는 목록 하나를 네 줄에 나눠 쓴 것입니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. 상자 4개가 든 목록. 정답 1, -1, -1, 1과는 아직 거리가 멉니다.

In [ ]:
xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

In [ ]:
check("L01-S47", globals())

## L01-S48 손실: 얼마나 틀렸나

영상: 챕터 15 「작은 데이터셋, 손실 함수」 [1:51:04부터 1:57:56까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6664&end=7076&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 손실(loss, 예측이 정답과 얼마나 다른지를 숫자 하나로 나타낸 것)을 만드는 장면 뒤에 새 셀에 2줄을 칩니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. 0보다 큰 숫자를 담은 `Value(data=...)` 하나. 보통 몇 정도(최대 16)입니다.

In [ ]:
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
loss

In [ ]:
check("L01-S48", globals())

## L01-S49 손실에서 거꾸로

영상: 챕터 15 「작은 데이터셋, 손실 함수」 [1:51:04부터 1:57:56까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=6664&end=7076&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 loss.backward()를 부르고 가중치 하나의 기울기를 들여다보는 장면 뒤에 새 셀에 3줄을 칩니다.

**보여야 하는 것**: 숫자는 무작위로 정해진 가중치에서 나오므로 사람마다, 실행할 때마다 다릅니다. 숫자 두 줄. 첫 줄(기울기)은 양수일 수도 음수일 수도 있고, 둘째 줄(값)은 -1과 1 사이입니다.

In [ ]:
loss.backward()
print(n.layers[0].neurons[0].w[0].grad)
print(n.layers[0].neurons[0].w[0].data)

In [ ]:
check("L01-S49", globals())

## L01-S50 파라미터 모으기 (긴 단계)

영상: 챕터 16 「파라미터 모으기」 [1:57:56부터 2:01:12까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7076&end=7272&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 1시간 57분 56초부터 봅니다. 고칠 숫자(파라미터: 가중치와 편향)를 한 목록으로 모으는 장면입니다. 세 셀에 2줄씩 넣습니다. (1) Neuron 셀(S42) 맨 아래 (2) Layer 셀(S44)의 return outs... 줄 아래(마지막 줄 Layer(2, 3)(x) 위) (3) MLP 셀(S45) 맨 아래. 세 셀을 차례로 다시 실행한 뒤 새 셀에 3줄을 칩니다. 영상에는 없는 줄 random.seed(1337)을 넣었습니다. 무작위 숫자를 늘 같은 순서로 뽑게 하는 줄이라, 이 화면의 숫자와 내 숫자를 똑같이 맞춰 볼 수 있습니다. 

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S42(Neuron 셀): Neuron 셀 맨 아래(return out 다음) · 2줄 넣기
- L01-S44(Layer 셀): Layer 셀의 return outs[0] if ... 줄 바로 아래 · 2줄 넣기
- L01-S45(MLP 셀): MLP 셀 맨 아래(return x 다음) · 2줄 넣기

**다시 실행할 셀(차례로)**: L01-S42(Neuron 셀) → L01-S44(Layer 셀) → L01-S45(MLP 셀)

**보여야 하는 것**: `41`

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S50: L01-S42(Neuron 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
import random
class Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1, 1))
  def __call__(self, x):
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  def parameters(self):
    return self.w + [self.b]

In [ ]:
# L01-S50: L01-S44(Layer 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class Layer:
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]
Layer(2, 3)(x)

In [ ]:
# L01-S50: L01-S45(MLP 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
class MLP:
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
random.seed(1337)
n = MLP(3, [4, 4, 1])
len(n.parameters())

In [ ]:
check("L01-S50", globals())

## L01-S51 한 걸음 학습 (긴 단계)

영상: 챕터 17 「경사하강법으로 직접 학습」 [2:01:12부터 2:14:03까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 2시간 1분 12초부터 봅니다. 앞으로 계산, 거꾸로 계산, 조금 고치기를 한 번 해 보는 장면 뒤에 새 셀에 6줄을 칩니다. S50 바로 다음에 실행하세요.

**보여야 하는 것**: `6.005...`처럼 6에 가까운 숫자 하나(S50의 시드를 그대로 썼을 때). 이 셀을 한 번 더 실행하면 손실이 조금 작아집니다.

In [ ]:
ypred = [n(x) for x in xs]
loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
loss.backward()
for p in n.parameters():
  p.data += -0.01 * p.grad
print(loss.data)

In [ ]:
check("L01-S51", globals())

## L01-S52 반복문으로 학습 (긴 단계)

영상: 챕터 17 「경사하강법으로 직접 학습」 [2:01:12부터 2:14:03까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 학습을 여러 번 되풀이하는 장면을 봅니다. 먼저 S50 셀과 S47 셀을 다시 실행해 신경망을 처음 상태로 되돌립니다. 그다음 새 셀에 7줄을 칩니다. 이 셀을 '학습 셀'이라고 부릅니다. 이 셀에는 영상처럼 일부러 한 가지가 빠져 있습니다. 결과를 보고 무엇이 이상한지 생각해 보세요.

**다시 실행할 셀(차례로)**: L01-S50 → L01-S47

**보여야 하는 것**: 0부터 19까지 20줄. 이 화면 기준으로 손실이 `0 6.005...`에서 시작해 몇 번 만에 0 가까이 떨어졌다가, k가 11일 때 한 번 튀고(`11 0.3356...`), 14부터는 4 근처로 뛰어올라 마지막 줄이 `19 3.975...`입니다. 컴퓨터에 따라 튀는 자리는 조금 다를 수 있습니다. `3.8050276695223e-11`처럼 e가 섞인 숫자는 아주 작은 수(0.000000000038...)를 짧게 쓴 것입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S52: 다시 실행할 L01-S50과(와) 같은 코드
random.seed(1337)
n = MLP(3, [4, 4, 1])
len(n.parameters())

In [ ]:
# L01-S52: 다시 실행할 L01-S47과(와) 같은 코드
xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

In [ ]:
for k in range(20):
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  loss.backward()
  for p in n.parameters():
    p.data += -0.05 * p.grad
  print(k, loss.data)

In [ ]:
check("L01-S52", globals())

## L01-S53 기울기 비우기(zero_grad)

영상: 챕터 17 「경사하강법으로 직접 학습」 [2:01:12부터 2:14:03까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 기울기를 0으로 비우는 줄을 넣는 장면을 봅니다. 학습 셀(S52)의 loss = 줄 바로 아래, loss.backward() 바로 위에 2줄을 넣습니다. 그다음 S50 셀, S47 셀, 학습 셀을 차례로 다시 실행합니다. 새 셀은 없습니다.

**앞 셀 고치기** (넣을 줄은 앱의 '따라 하기' 화면에 있습니다. 그 셀을 그 자리에서 고칩니다)
- L01-S52(학습 셀): 학습 셀의 loss = 줄 바로 아래(loss.backward() 위) · 2줄 넣기

**다시 실행할 셀(차례로)**: L01-S50 → L01-S47 → L01-S52(학습 셀)

**보여야 하는 것**: 학습 셀의 20줄에서 손실이 6.0에서 시작해 거의 매번 줄어들고, 마지막 줄은 `19 0.0596...`처럼 0.06 근처입니다.

정답 노트북에는 다시 실행할 셀을 이 자리에 다시 두었습니다(이 단계에서 고친 셀은 고친 뒤의 전체).

In [ ]:
# L01-S53: 다시 실행할 L01-S50과(와) 같은 코드
random.seed(1337)
n = MLP(3, [4, 4, 1])
len(n.parameters())

In [ ]:
# L01-S53: 다시 실행할 L01-S47과(와) 같은 코드
xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

In [ ]:
# L01-S53: L01-S52(학습 셀)을(를) 이 단계에서 고친 뒤의 전체(다시 실행)
for k in range(20):
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  for p in n.parameters():
    p.data += -0.05 * p.grad
  print(k, loss.data)

In [ ]:
check("L01-S53", globals())

## L01-S54 예측 보기

영상: 챕터 17 「경사하강법으로 직접 학습」 [2:01:12부터 2:14:03까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=7272&end=8043&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 같은 구간에서 학습 뒤 예측을 확인하는 장면 뒤에 새 셀에 1줄을 칩니다.

**보여야 하는 것**: 상자 4개: 약 `0.86`, `-0.98`, `-0.86`, `0.86`. 정답 1, -1, -1, 1에 가까워졌습니다.

In [ ]:
ypred

In [ ]:
check("L01-S54", globals())

## L01-S55 배운 것 정리

영상: 챕터 18 「배운 것 요약, 현대 신경망으로 가는 길」 [2:14:03부터 2:16:46까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=8043&end=8206&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 2시간 14분 3초부터 요약 부분을 봅니다. 그다음 코드 없이 아래 문장의 빈칸 넷을 내 말로 채워 앱의 메모 칸에 적습니다. '학습 한 바퀴 = (   ) → (   ) → (   ) → (   )'. 그리고 오늘 배운 것을 한 줄로 요약해 적습니다.

**보여야 하는 것**: 빈칸 넷과 한 줄 요약이 메모 칸에 저장되면 끝입니다. 정답 예: 앞으로 계산해 손실 구하기 → 기울기 비우기 → 거꾸로 계산(backward) → 기울기 반대로 조금 고치기.

이 단계는 노트북에서 칠 코드가 없습니다. 앱 화면에서 진행합니다.

## L01-S56 진짜 micrograd 둘러보기

영상: 챕터 19 「GitHub의 micrograd 전체 코드 둘러보기」 [2:16:46부터 2:21:10까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=8206&end=8470&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 영상 2시간 16분 46초부터 봅니다. 그다음 실제 micrograd 저장소의 두 파일을 엽니다. engine.py: https://github.com/karpathy/micrograd/blob/master/micrograd/engine.py , nn.py: https://github.com/karpathy/micrograd/blob/master/micrograd/nn.py . 내 코드와 다른 점 3가지를 찾아 체크합니다. 예: relu라는 다른 활성 함수, zero_grad를 함수로 만들어 둔 것, Module이라는 공통 부모 설계도. 코드는 치지 않습니다.

**보여야 하는 것**: 다른 점 3칸이 체크되면 끝입니다.

이 단계는 노트북에서 칠 코드가 없습니다. 앱 화면에서 진행합니다.

## L01-S57 PyTorch 속 tanh 기울기 찾기

영상: 챕터 20 「PyTorch의 tanh backward 찾아보기」 [2:21:10부터 2:24:39까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=8470&end=8679&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 선택 단계입니다. 영상 2시간 21분 10초부터 봅니다. PyTorch처럼 큰 저장소에서 tanh의 backward 식을 찾아가는 장면입니다. 코드는 치지 않습니다.

**보여야 하는 것**: 화면에 칠 것은 없습니다.

이 단계는 노트북에서 칠 코드가 없습니다. 앱 화면에서 진행합니다.

## L01-S58 결론과 NG 장면

영상: 챕터 21 「결론」 [2:24:39부터 2:25:20까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=8679&end=8720&cc_load_policy=1&cc_lang_pref=ko&hl=ko) · 챕터 22 「NG 장면(outtakes)」 [2:25:20부터 2:25:52까지 보기](https://www.youtube-nocookie.com/embed/VMj-3S1tku0?start=8720&end=8752&cc_load_policy=1&cc_lang_pref=ko&hl=ko)

**입력할 것**: 선택 단계입니다. 영상 2시간 24분 39초부터 끝까지(결론, NG 장면) 봅니다.

**보여야 하는 것**: 화면에 칠 것은 없습니다. 1편을 끝냈습니다.

이 단계는 노트북에서 칠 코드가 없습니다. 앱 화면에서 진행합니다.

## 재구성(빈 셀에서 다시 만들기)

본문을 마친 뒤(다른 날도 됩니다) 앞 단계를 보지 않고 다시 만들어 보는 연습입니다. 조각은 3개(L01-R1, L01-R2, L01-R3)입니다.

- 정본 과제는 빈 노트북에서 시작합니다. 이 노트북에서는 아래 '재구성 시작' 셀이 빈 노트북을 대신합니다. 위 본문에서 만든 이름(Value 같은 것과 math 같은 불러온 도구까지)을 따로 보관하고 지웁니다. 채점 도구와 채점 기록은 그대로입니다. 그래서 `import math` 같은 줄도 다시 씁니다. 이 셀을 실행한 뒤에는 위 본문 셀을 다시 실행하지 않습니다.
- 조각마다 확인 셀(정본 check_code)이 미리 들어 있습니다. 내 셀을 실행한 뒤 확인 셀로 출력을 보고, 채점 셀로 기록합니다.
- 힌트는 앱의 조각 화면에서 봅니다. 힌트 H2까지 보고 완성하면 통과로 칩니다. 노트북은 힌트를 세지 않습니다.
- 다른 날 이어서 할 때: 맨 위 준비 셀 두 개 → '재구성 시작' 셀 → 조각 순서대로 → 마무리 셀.
- 재구성 결과도 마무리 셀의 결과코드에 들어갑니다.

### 재구성 시작

아래 셀을 실행하면 위에서 만든 이름을 따로 보관하고 지워 빈 노트북처럼 만듭니다. 보관한 이름은 이 편의 노트북 상태에서 시작하는 조각이 되돌려 씁니다.

In [ ]:
# 재구성 시작 셀: 고치지 말고 실행만 하세요. 위 본문에서 만든 이름을 따로 보관하고 지워 빈 노트북처럼 만듭니다.
import z2h_check.rebuild as _z2h_rebuild
_z2h_rebuild.start(globals())

## L01-R1 +, *, tanh와 각 _backward(+= 누적)가 있는 Value

**과제**: Colab이나 맥미니 Jupyter에서 빈 노트북을 하나 엽니다. 앞 단계를 보지 않고, 새 셀에 Value 설계도를 다시 만듭니다. 들어가야 하는 것: __init__(data, _children, _op, label, grad, _backward), __repr__, __add__, __mul__, tanh. 세 연산 모두 _backward를 갖고, 기울기는 +=로 쌓아야 합니다. 맨 위에 import math를 잊지 마세요. 다 만들면 확인 셀(check_code)을 붙여 넣어 실행합니다. backward()는 R2에서 만들므로 여기서는 없어도 됩니다.

**통과 기준**: 확인 셀의 출력이 `[-1.5, 1.0, 0.5, 0.0, 0.5]`와 `True 6.0` 두 줄이면 통과입니다.

**실행 순서**: 아래 빈 셀에 작성하고 실행 → 확인 셀 → 채점 셀.

In [ ]:
import math
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

In [ ]:
x1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); b = Value(6.8813735870195432)
a1 = x1*w1; a2 = x2*w2; s = a1 + a2; n = s + b; o = n.tanh()
o.grad = 1.0
for v in (o, n, s, a2, a1):
  v._backward()
print([round(v.grad, 4) for v in (x1, w1, x2, w2, b)])
c = Value(3.0); d = c * c; d.grad = 1.0; d._backward()
print(d._prev == {c}, c.grad)

In [ ]:
check("L01-R1", globals())

## L01-R2 위상 정렬을 쓰는 backward()

**과제**: R1에서 만든 Value 셀 맨 아래에, 앞 단계를 보지 않고 backward()를 다시 만듭니다. 줄 세우기(위상 정렬) → 맨 끝 기울기 1 → 줄을 거꾸로 돌며 _backward 부르기. Value 셀을 다시 실행한 뒤 확인 셀을 붙여 넣어 실행합니다.

**통과 기준**: 확인 셀의 출력이 `2.0`과 `[-1.5, 1.0, 0.5, 0.0, 0.5]` 두 줄이면 통과입니다.

**실행 순서**: 위 L01-R1 셀 맨 아래에 덧붙이고 그 셀을 다시 실행 → 아래 확인 셀 → 채점 셀. 새 빈 셀은 없습니다.

정답 노트북에는 덧붙인 뒤의 L01-R1 셀 전체를 이 자리에 다시 두었습니다.

In [ ]:
import math
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label
  def __repr__(self):
    return f"Value(data={self.data})"
  def __add__(self, other):
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out
  def __mul__(self, other):
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out
  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
a = Value(3.0)
b = a + a
b.backward()
print(a.grad)
x1 = Value(2.0); x2 = Value(0.0); w1 = Value(-3.0); w2 = Value(1.0); bb = Value(6.8813735870195432)
o = (x1*w1 + x2*w2 + bb).tanh()
o.backward()
print([round(v.grad, 4) for v in (x1, w1, x2, w2, bb)])

In [ ]:
check("L01-R2", globals())

## L01-R3 학습 반복문(앞으로, 비우기, 거꾸로, 고치기)

**과제**: 이 편의 노트북에서 Value 셀, Neuron, Layer, MLP 셀(S50에서 고친 것), 데이터 셀(S47)까지 실행된 상태로 시작합니다. 새 셀에 앞 단계를 보지 않고, random.seed(1337)과 n = MLP(3, [4, 4, 1])로 신경망을 새로 만든 뒤 학습률 0.05로 20번 학습하는 반복문을 씁니다. 반복마다 print(k, loss.data)로 손실을 찍습니다. 그다음 확인 셀을 실행합니다.

**통과 기준**: 확인 셀의 끝이 `True`이면 통과입니다(마지막 손실 < 3.0). 시드와 학습률을 위처럼 했다면 숫자는 `0.0596`입니다.

**실행 순서**: 아래 'L01-R3 시작' 셀(재구성 시작 셀이 보관한 본문 이름을 되돌림) → 빈 셀에 작성하고 실행 → 확인 셀 → 채점 셀. 다른 날 새로 열었다면 '시작' 셀이 먼저 실행할 본문 셀을 알려 줍니다.

In [ ]:
# L01-R3 시작 셀: 고치지 말고 실행만 하세요. 재구성 시작 셀이 보관한 본문 이름을 되돌립니다.
import z2h_check.rebuild as _z2h_rebuild
_z2h_rebuild.resume(globals(), "L01-R3")

In [ ]:
random.seed(1337)
n = MLP(3, [4, 4, 1])
for k in range(20):
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  for p in n.parameters():
    p.data += -0.05 * p.grad
  print(k, loss.data)

In [ ]:
print(round(loss.data, 4), loss.data < 3.0)

In [ ]:
check("L01-R3", globals())

## 마무리: 채점표와 결과코드

아래 셀을 실행하면 채점표와 결과코드가 나옵니다. 맨 아래 줄의 결과코드를 앱의 '결과코드 붙여넣기' 칸에 붙여 넣으세요. 로컬 Jupyter에서 진도 서버가 켜져 있으면 결과를 서버로도 보내 보고, Colab에서는 "서버에 닿지 못했습니다"가 정상입니다.

재구성 조각(L01-R1~R3)의 결과도 같은 결과코드에 들어갑니다. 아직 하지 않았다면 그 칸은 "아직 안 함"으로 보이고, 그대로 붙여 넣어도 됩니다.

In [ ]:
# 마지막 칸: 채점표와 결과코드를 보여 줍니다. 맨 아래 줄(Z2H1. 로 시작)을 앱의 '결과코드 붙여넣기' 칸에 넣으세요.
result_code = finish("L01")